In [ ]:
#@title Cell 23.1 - Notebook overview
from IPython.display import display, Markdown

display(Markdown(r"""
# Notebook 23: Targeted AMR-Sequence Extraction

## Purpose

Notebook 23 will extract and save targeted AMR-related nucleotide sequences
from the NCBI assemblies of the Model C pathogens.

It will:

1. start with the 9,377 Model 3B pathogens;
2. retain the 9,058 pathogens with assembly accessions;
3. exclude the 319 pathogens without assemblies from Model C;
4. validate the 9,058 assembly accessions;
5. construct the agreed panel of 297 AMR-related loci;
6. download each pathogen assembly;
7. extract sequences for:

   - 266 AMR-gene loci using AMRFinderPlus;
   - 30 chromosomal coding genes using the corresponding MG1655 gene
     sequences as references; and
   - one 100-nucleotide `blaTEMp` upstream region;

8. record whether each sequence was extracted, missing, partial, disrupted or
   uncertain;
9. save the results in validated batches of 10 assemblies;
10. delete each downloaded genome after its batch has been safely saved; and
11. allow extraction to resume from the first unfinished batch.

## Model boundary

The existing Model 3B notebooks and their 9,377-pathogen results will not be
changed.

Notebook 23 only extracts and saves the targeted nucleotide sequences. It does
not align sequences, compare nucleotide positions or calculate
\(K_{\mathrm{seq}}\). Those steps belong to Notebook 24.

Whole-genome similarity is outside the scope of this notebook.

## Expected notebook length

Notebook 23 contains **11 cells**.
"""))

print(
    "Transition: The next cell will import the required packages "
    "and define the Notebook 23 settings."
)


# Notebook 23: Targeted AMR-Sequence Extraction

## Purpose

Notebook 23 will extract and save targeted AMR-related nucleotide sequences
from the NCBI assemblies of the Model C pathogens.

It will:

1. start with the 9,377 Model 3B pathogens;
2. retain the 9,058 pathogens with assembly accessions;
3. exclude the 319 pathogens without assemblies from Model C;
4. validate the 9,058 assembly accessions;
5. construct the agreed panel of 297 AMR-related loci;
6. download each pathogen assembly;
7. extract sequences for:

   - 266 AMR-gene loci using AMRFinderPlus;
   - 30 chromosomal coding genes using the corresponding MG1655 gene
     sequences as references; and
   - one 100-nucleotide `blaTEMp` upstream region;

8. record whether each sequence was extracted, missing, partial, disrupted or
   uncertain;
9. save the results in validated batches of 10 assemblies;
10. delete each downloaded genome after its batch has been safely saved; and
11. allow extraction to resume from the first unfinished batch.

## Model boundary

The existing Model 3B notebooks and their 9,377-pathogen results will not be
changed.

Notebook 23 only extracts and saves the targeted nucleotide sequences. It does
not align sequences, compare nucleotide positions or calculate
\(K_{\mathrm{seq}}\). Those steps belong to Notebook 24.

Whole-genome similarity is outside the scope of this notebook.

## Expected notebook length

Notebook 23 contains **11 cells**.


Transition: The next cell will import the required packages and define the Notebook 23 settings.


In [ ]:
#@title Cell 23.2 - Import packages and define notebook settings
from pathlib import Path
import json
import shutil
import subprocess
import time
import zipfile

import numpy as np
import pandas as pd


# Expected cohort sizes from Notebook 22.
EXPECTED_TOTAL_BIOSAMPLES = 9377
EXPECTED_MODEL_C_BIOSAMPLES = 9058
EXPECTED_EXCLUDED_BIOSAMPLES = 319

# Existing Model 3B sequence-related feature counts.
EXPECTED_FULL_GENE_LABELS = 267
EXPECTED_POINT_TARGET_LOCI = 25

# Required input archives.
ASSEMBLY_AUDIT_ARCHIVE_NAME = (
    "22_ecoli_assembly_availability_audit_outputs.zip"
)

MODEL_3B_ARCHIVE_NAME = (
    "15B_full_gene_genomic_antibiotic_kernels_outputs.zip"
)

# Notebook 23 working directories.
WORK_DIRECTORY = Path("/content/notebook23_work")
AUDIT_DIRECTORY = WORK_DIRECTORY / "notebook22"
MODEL_3B_DIRECTORY = WORK_DIRECTORY / "notebook15B"
NCBI_DIRECTORY = WORK_DIRECTORY / "ncbi"
DOWNLOAD_DIRECTORY = NCBI_DIRECTORY / "downloads"
CHECKPOINT_DIRECTORY = WORK_DIRECTORY / "checkpoints"
RESULTS_DIRECTORY = WORK_DIRECTORY / "results"

for directory in [
    WORK_DIRECTORY,
    AUDIT_DIRECTORY,
    MODEL_3B_DIRECTORY,
    NCBI_DIRECTORY,
    DOWNLOAD_DIRECTORY,
    CHECKPOINT_DIRECTORY,
    RESULTS_DIRECTORY,
]:
    directory.mkdir(parents=True, exist_ok=True)

# Notebook 23 output filenames.
MODEL_C_MANIFEST_FILENAME = (
    "23_model_c_assembly_manifest.csv"
)

ACCESSION_VALIDATION_FILENAME = (
    "23_current_assembly_validation.csv"
)

TARGET_LOCUS_PANEL_FILENAME = (
    "23_target_amr_locus_panel.csv"
)

TARGET_SEQUENCE_FILENAME = (
    "23_targeted_amr_sequences.csv"
)

EXTRACTION_SUMMARY_FILENAME = (
    "23_sequence_extraction_summary.json"
)

OUTPUT_ARCHIVE_NAME = (
    "23_targeted_amr_sequence_data_outputs.zip"
)

print("Notebook 23 settings initialised.")
print(f"Model C starting cohort: {EXPECTED_MODEL_C_BIOSAMPLES:,} pathogens")

print(
    "\nTransition: The next cell will locate or upload "
    "the Notebook 22 and Notebook 15B archives."
)

Notebook 23 settings initialised.
Model C starting cohort: 9,058 pathogens

Transition: The next cell will locate or upload the Notebook 22 and Notebook 15B archives.


In [ ]:
#@title Cell 23.3 - Load the required input archives
from google.colab import drive, files

drive.mount("/content/drive")

DRIVE_REFERENCE_DIRECTORY = Path(
    "/content/drive/MyDrive/Model3_MIC_Project/reference_files"
)
DRIVE_REFERENCE_DIRECTORY.mkdir(parents=True, exist_ok=True)

input_archive_paths = {
    ASSEMBLY_AUDIT_ARCHIVE_NAME:
        DRIVE_REFERENCE_DIRECTORY / ASSEMBLY_AUDIT_ARCHIVE_NAME,

    MODEL_3B_ARCHIVE_NAME:
        DRIVE_REFERENCE_DIRECTORY / MODEL_3B_ARCHIVE_NAME,
}

missing_archives = [
    name
    for name, path in input_archive_paths.items()
    if not path.exists()
]

if missing_archives:
    print("Please upload:\n")
    for name in missing_archives:
        print(f"- {name}")

    uploaded_files = files.upload()

    for archive_name in missing_archives:
        if archive_name not in uploaded_files:
            raise FileNotFoundError(
                f"Required archive was not uploaded: {archive_name}"
            )

        source_path = Path("/content") / archive_name
        destination_path = input_archive_paths[archive_name]

        shutil.copy2(source_path, destination_path)
        print(f"Saved to Google Drive: {archive_name}")
else:
    print("Both required archives were found in Google Drive.")

for archive_name, archive_path in input_archive_paths.items():
    if not zipfile.is_zipfile(archive_path):
        raise ValueError(f"Invalid ZIP archive: {archive_name}")

for extraction_directory in [
    AUDIT_DIRECTORY,
    MODEL_3B_DIRECTORY,
]:
    if extraction_directory.exists():
        shutil.rmtree(extraction_directory)

    extraction_directory.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(
    input_archive_paths[ASSEMBLY_AUDIT_ARCHIVE_NAME],
    mode="r",
) as archive:
    archive.extractall(AUDIT_DIRECTORY)

with zipfile.ZipFile(
    input_archive_paths[MODEL_3B_ARCHIVE_NAME],
    mode="r",
) as archive:
    archive.extractall(MODEL_3B_DIRECTORY)

print("Both archives were extracted successfully.")

print(
    "\nTransition: The next cell will load the Notebook 22 audit "
    "and construct the 9,058-pathogen Model C manifest."
)

Mounted at /content/drive
Both required archives were found in Google Drive.
Both archives were extracted successfully.

Transition: The next cell will load the Notebook 22 audit and construct the 9,058-pathogen Model C manifest.


In [ ]:
#@title Cell 23.4 - Construct the Model C pathogen manifest
# This cell retains the 9,058 pathogens with one assembly accession and
# preserves their original Model 3B row positions.

audit_matches = list(
    AUDIT_DIRECTORY.rglob(
        "22_biosample_assembly_availability_audit.csv"
    )
)

if len(audit_matches) != 1:
    raise FileNotFoundError(
        "Expected exactly one Notebook 22 assembly-audit file; "
        f"found {len(audit_matches)}."
    )

assembly_audit = pd.read_csv(audit_matches[0])

required_columns = {
    "biosample",
    "assembly_count",
    "assembly_accessions",
}

missing_columns = required_columns.difference(
    assembly_audit.columns
)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

if assembly_audit["biosample"].duplicated().any():
    raise ValueError(
        "The Notebook 22 audit contains duplicate BioSamples."
    )

assembly_audit["biosample"] = (
    assembly_audit["biosample"]
    .astype(str)
    .str.strip()
    .str.upper()
)

assembly_audit["assembly_count"] = pd.to_numeric(
    assembly_audit["assembly_count"],
    errors="raise",
).astype(int)

assembly_audit["assembly_accessions"] = (
    assembly_audit["assembly_accessions"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)

assembly_audit["model_3b_row_index"] = np.arange(
    len(assembly_audit)
)

available_mask = assembly_audit["assembly_count"] == 1
excluded_mask = assembly_audit["assembly_count"] == 0
ambiguous_mask = assembly_audit["assembly_count"] > 1

available_count = int(available_mask.sum())
excluded_count = int(excluded_mask.sum())
ambiguous_count = int(ambiguous_mask.sum())

if len(assembly_audit) != EXPECTED_TOTAL_BIOSAMPLES:
    raise ValueError(
        f"Expected {EXPECTED_TOTAL_BIOSAMPLES:,} BioSamples; "
        f"found {len(assembly_audit):,}."
    )

if available_count != EXPECTED_MODEL_C_BIOSAMPLES:
    raise ValueError(
        f"Expected {EXPECTED_MODEL_C_BIOSAMPLES:,} pathogens "
        f"with one assembly; found {available_count:,}."
    )

if excluded_count != EXPECTED_EXCLUDED_BIOSAMPLES:
    raise ValueError(
        f"Expected {EXPECTED_EXCLUDED_BIOSAMPLES:,} pathogens "
        f"without assemblies; found {excluded_count:,}."
    )

if ambiguous_count != 0:
    raise ValueError(
        f"Found {ambiguous_count:,} ambiguous assembly mappings."
    )

model_c_manifest = (
    assembly_audit.loc[
        available_mask,
        [
            "model_3b_row_index",
            "biosample",
            "assembly_accessions",
        ],
    ]
    .rename(
        columns={
            "assembly_accessions":
                "reported_assembly_accession"
        }
    )
    .reset_index(drop=True)
)

model_c_manifest.insert(
    0,
    "model_c_row_index",
    np.arange(len(model_c_manifest)),
)

if (
    model_c_manifest["reported_assembly_accession"]
    .eq("")
    .any()
):
    raise ValueError(
        "A retained pathogen has an empty assembly accession."
    )

manifest_path = (
    RESULTS_DIRECTORY / MODEL_C_MANIFEST_FILENAME
)

model_c_manifest.to_csv(
    manifest_path,
    index=False,
)

cohort_summary = pd.DataFrame(
    {
        "group": [
            "All Model 3B pathogens",
            "Retained for Model C",
            "Excluded: no assembly",
            "Ambiguous assembly mapping",
        ],
        "pathogens": [
            len(assembly_audit),
            len(model_c_manifest),
            excluded_count,
            ambiguous_count,
        ],
    }
)

display(cohort_summary)
display(model_c_manifest.head())

print(f"Saved: {manifest_path}")
print(
    "\nTransition: The next cell will install and test "
    "the NCBI command-line tools."
)

,group,pathogens
0,All Model 3B pathogens,9377
1,Retained for Model C,9058
2,Excluded: no assembly,319
3,Ambiguous assembly mapping,0


,model_c_row_index,model_3b_row_index,biosample,reported_assembly_accession
0,0,0,SAMN02138598,GCA_000522345.1
1,1,1,SAMN02138602,GCA_000522325.1
2,2,2,SAMN02138649,GCA_000522145.1
3,3,3,SAMN02138668,GCA_000492275.1
4,4,4,SAMN02581257,GCA_000692795.1


Saved: /content/notebook23_work/results/23_model_c_assembly_manifest.csv

Transition: The next cell will install and test the NCBI command-line tools.


In [ ]:
#@title Cell 23.5 - Install and test the NCBI command-line tools
# This cell installs the official NCBI Datasets tools and confirms that
# Colab can retrieve genome-assembly metadata from NCBI.

import os
import platform

if platform.machine().lower() not in {
    "x86_64",
    "amd64",
}:
    raise RuntimeError(
        f"Unsupported Colab processor: {platform.machine()}"
    )

TOOLS_DIRECTORY = WORK_DIRECTORY / "bin"
TOOLS_DIRECTORY.mkdir(parents=True, exist_ok=True)

DATASETS_EXECUTABLE = TOOLS_DIRECTORY / "datasets"
DATAFORMAT_EXECUTABLE = TOOLS_DIRECTORY / "dataformat"

tool_urls = {
    DATASETS_EXECUTABLE:
        "https://ftp.ncbi.nlm.nih.gov/pub/datasets/"
        "command-line/v2/linux-amd64/datasets",

    DATAFORMAT_EXECUTABLE:
        "https://ftp.ncbi.nlm.nih.gov/pub/datasets/"
        "command-line/v2/linux-amd64/dataformat",
}

for executable_path, download_url in tool_urls.items():
    if not executable_path.exists():
        subprocess.run(
            [
                "curl",
                "--fail",
                "--location",
                "--output",
                str(executable_path),
                download_url,
            ],
            check=True,
        )

    os.chmod(executable_path, 0o755)

datasets_version_result = subprocess.run(
    [str(DATASETS_EXECUTABLE), "--version"],
    capture_output=True,
    text=True,
    check=True,
)

print(datasets_version_result.stdout.strip())

dataformat_test_result = subprocess.run(
    [str(DATAFORMAT_EXECUTABLE), "--help"],
    capture_output=True,
    text=True,
    check=True,
)

if "dataformat" not in dataformat_test_result.stdout.lower():
    raise RuntimeError(
        "The dataformat installation test failed."
    )

print("dataformat: installed and responsive")

# Use E. coli K-12 MG1655 only to test NCBI access.
test_accession = "GCF_000005845.2"

test_result = subprocess.run(
    [
        str(DATASETS_EXECUTABLE),
        "summary",
        "genome",
        "accession",
        test_accession,
        "--as-json-lines",
    ],
    capture_output=True,
    text=True,
    timeout=120,
)

if test_result.returncode != 0:
    raise RuntimeError(
        "NCBI connection test failed:\n"
        + test_result.stderr.strip()
    )

test_lines = [
    line
    for line in test_result.stdout.splitlines()
    if line.strip()
]

if not test_lines:
    raise RuntimeError(
        "NCBI returned no metadata for the test assembly."
    )

json.loads(test_lines[0])

print(
    f"NCBI connection confirmed with test assembly "
    f"{test_accession}."
)

print(
    "\nTransition: The next cell will validate the 9,058 "
    "reported assembly accessions in restartable batches."
)

datasets version: 18.33.1
dataformat: installed and responsive
NCBI connection confirmed with test assembly GCF_000005845.2.

Transition: The next cell will validate the 9,058 reported assembly accessions in restartable batches.


In [ ]:
#@title Cell 23.6 - Retrieve current NCBI assembly metadata
# This cell queries the 9,058 reported assembly accessions in restartable
# batches and saves the returned NCBI metadata in Google Drive.

import hashlib

VALIDATION_BATCH_SIZE = 250
MAX_QUERY_ATTEMPTS = 3
RETRY_WAIT_SECONDS = 10

DRIVE_NOTEBOOK23_DIRECTORY = Path(
    "/content/drive/MyDrive/Model3_MIC_Project/notebook23"
)

DRIVE_VALIDATION_DIRECTORY = (
    DRIVE_NOTEBOOK23_DIRECTORY
    / "accession_validation_batches"
)

DRIVE_VALIDATION_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

reported_accessions = (
    model_c_manifest["reported_assembly_accession"]
    .astype(str)
    .str.strip()
    .str.upper()
)

invalid_accession_mask = ~reported_accessions.str.fullmatch(
    r"GC[AF]_\d+\.\d+"
)

if invalid_accession_mask.any():
    invalid_values = (
        reported_accessions[invalid_accession_mask]
        .drop_duplicates()
        .tolist()
    )

    raise ValueError(
        f"Invalid assembly accession format: {invalid_values[:10]}"
    )

# An assembly could theoretically be linked to more than one BioSample.
# It only needs to be queried once.
unique_accessions = list(
    dict.fromkeys(reported_accessions.tolist())
)

number_of_batches = (
    len(unique_accessions)
    + VALIDATION_BATCH_SIZE
    - 1
) // VALIDATION_BATCH_SIZE

print(f"Model C pathogens: {len(model_c_manifest):,}")
print(f"Unique assembly accessions: {len(unique_accessions):,}")
print(f"Validation batches: {number_of_batches:,}")

completed_batches = 0
returned_metadata_records = 0

for batch_number in range(number_of_batches):
    start = batch_number * VALIDATION_BATCH_SIZE
    stop = min(
        start + VALIDATION_BATCH_SIZE,
        len(unique_accessions),
    )

    batch_accessions = unique_accessions[start:stop]
    batch_label = f"{batch_number + 1:03d}"

    input_text = "\n".join(batch_accessions) + "\n"
    input_hash = hashlib.sha256(
        input_text.encode("utf-8")
    ).hexdigest()

    input_path = (
        DRIVE_VALIDATION_DIRECTORY
        / f"batch_{batch_label}_accessions.txt"
    )

    report_path = (
        DRIVE_VALIDATION_DIRECTORY
        / f"batch_{batch_label}_report.jsonl"
    )

    status_path = (
        DRIVE_VALIDATION_DIRECTORY
        / f"batch_{batch_label}_status.json"
    )

    batch_already_complete = False

    if report_path.exists() and status_path.exists():
        try:
            saved_status = json.loads(
                status_path.read_text(encoding="utf-8")
            )

            saved_lines = [
                line
                for line in report_path.read_text(
                    encoding="utf-8"
                ).splitlines()
                if line.strip()
            ]

            for line in saved_lines:
                json.loads(line)

            batch_already_complete = (
                saved_status.get("completed") is True
                and saved_status.get("input_sha256")
                == input_hash
                and saved_status.get("input_count")
                == len(batch_accessions)
            )

        except Exception:
            batch_already_complete = False

    if batch_already_complete:
        completed_batches += 1
        returned_metadata_records += len(saved_lines)

        print(
            f"Batch {batch_label}/{number_of_batches:03d}: "
            f"already complete"
        )
        continue

    input_path.write_text(
        input_text,
        encoding="utf-8",
    )

    query_succeeded = False
    last_error = ""

    for attempt in range(1, MAX_QUERY_ATTEMPTS + 1):
        query_result = subprocess.run(
            [
                str(DATASETS_EXECUTABLE),
                "summary",
                "genome",
                "accession",
                "--inputfile",
                str(input_path),
                "--assembly-version",
                "all",
                "--as-json-lines",
            ],
            capture_output=True,
            text=True,
            timeout=600,
        )

        if query_result.returncode == 0:
            report_lines = [
                line
                for line in query_result.stdout.splitlines()
                if line.strip()
            ]

            for line in report_lines:
                json.loads(line)

            temporary_report_path = report_path.with_suffix(
                ".jsonl.tmp"
            )

            temporary_report_path.write_text(
                "\n".join(report_lines)
                + ("\n" if report_lines else ""),
                encoding="utf-8",
            )

            temporary_report_path.replace(report_path)

            batch_status = {
                "batch_number": batch_number + 1,
                "completed": True,
                "input_count": len(batch_accessions),
                "returned_metadata_records": len(report_lines),
                "input_sha256": input_hash,
            }

            temporary_status_path = status_path.with_suffix(
                ".json.tmp"
            )

            temporary_status_path.write_text(
                json.dumps(
                    batch_status,
                    indent=2,
                ),
                encoding="utf-8",
            )

            temporary_status_path.replace(status_path)

            completed_batches += 1
            returned_metadata_records += len(report_lines)
            query_succeeded = True

            print(
                f"Batch {batch_label}/{number_of_batches:03d}: "
                f"{len(batch_accessions)} accessions submitted; "
                f"{len(report_lines)} metadata records returned"
            )

            break

        last_error = (
            query_result.stderr.strip()
            or query_result.stdout.strip()
            or "Unknown NCBI error"
        )

        if attempt < MAX_QUERY_ATTEMPTS:
            time.sleep(RETRY_WAIT_SECONDS * attempt)

    if not query_succeeded:
        raise RuntimeError(
            f"Batch {batch_label} failed after "
            f"{MAX_QUERY_ATTEMPTS} attempts:\n{last_error}"
        )

print("\nNCBI batch retrieval completed.")
print(f"Completed batches: {completed_batches:,}")
print(
    f"Metadata records returned: "
    f"{returned_metadata_records:,}"
)

print(
    "\nTransition: The next cell will match these records "
    "to the 9,058 pathogens and resolve current accessions."
)

Model C pathogens: 9,058
Unique assembly accessions: 9,058
Validation batches: 37
Batch 001/037: already complete
Batch 002/037: already complete
Batch 003/037: already complete
Batch 004/037: already complete
Batch 005/037: already complete
Batch 006/037: already complete
Batch 007/037: already complete
Batch 008/037: already complete
Batch 009/037: already complete
Batch 010/037: already complete
Batch 011/037: already complete
Batch 012/037: already complete
Batch 013/037: already complete
Batch 014/037: already complete
Batch 015/037: already complete
Batch 016/037: already complete
Batch 017/037: already complete
Batch 018/037: already complete
Batch 019/037: already complete
Batch 020/037: already complete
Batch 021/037: already complete
Batch 022/037: already complete
Batch 023/037: already complete
Batch 024/037: already complete
Batch 025/037: already complete
Batch 026/037: already complete
Batch 027/037: already complete
Batch 028/037: already complete
Batch 029/037: already

In [ ]:
#@title Cell 23.7 - Resolve current assembly accessions
# This cell converts the NCBI metadata to a table and matches one current
# assembly accession to each of the 9,058 Model C pathogens.

import io
import re

batch_report_paths = sorted(
    DRIVE_VALIDATION_DIRECTORY.glob(
        "batch_*_report.jsonl"
    )
)

if len(batch_report_paths) != number_of_batches:
    raise ValueError(
        f"Expected {number_of_batches} completed batch reports; "
        f"found {len(batch_report_paths)}."
    )

combined_report_path = (
    CHECKPOINT_DIRECTORY
    / "23_all_assembly_metadata.jsonl"
)

with combined_report_path.open(
    mode="w",
    encoding="utf-8",
) as combined_file:

    for report_path in batch_report_paths:
        report_text = report_path.read_text(
            encoding="utf-8"
        )

        if report_text:
            combined_file.write(report_text)

metadata_fields = ",".join(
    [
        "accession",
        "current-accession",
        "assminfo-status",
        "assminfo-suppression-reason",
        "assminfo-level",
        "assminfo-biosample-accession",
        "organism-name",
        "organism-tax-id",
        "assminfo-paired-assm-accession",
        "assminfo-paired-assm-status",
        "source_database",
    ]
)

dataformat_result = subprocess.run(
    [
        str(DATAFORMAT_EXECUTABLE),
        "tsv",
        "genome",
        "--inputfile",
        str(combined_report_path),
        "--fields",
        metadata_fields,
        "--elide-header",
        "--force",
    ],
    capture_output=True,
    text=True,
    timeout=600,
)

if dataformat_result.returncode != 0:
    raise RuntimeError(
        "NCBI metadata conversion failed:\n"
        + dataformat_result.stderr.strip()
    )

metadata_columns = [
    "accession",
    "current_accession",
    "assembly_status",
    "suppression_reason",
    "assembly_level",
    "biosample",
    "organism_name",
    "organism_tax_id",
    "paired_accession",
    "paired_status",
    "source_database",
]

assembly_metadata = pd.read_csv(
    io.StringIO(dataformat_result.stdout),
    sep="\t",
    names=metadata_columns,
    dtype=str,
    keep_default_na=False,
)

assembly_metadata = (
    assembly_metadata
    .drop_duplicates()
    .reset_index(drop=True)
)

for column in metadata_columns:
    assembly_metadata[column] = (
        assembly_metadata[column]
        .astype(str)
        .str.strip()
    )

for column in [
    "accession",
    "current_accession",
    "biosample",
    "paired_accession",
]:
    assembly_metadata[column] = (
        assembly_metadata[column].str.upper()
    )

accession_pattern = re.compile(
    r"GC[AF]_\d+\.\d+"
)

assembly_metadata["accession_root"] = (
    assembly_metadata["accession"]
    .str.rsplit(".", n=1)
    .str[0]
)

records_by_accession = {
    accession: group.reset_index(drop=True)
    for accession, group in assembly_metadata.groupby(
        "accession",
        sort=False,
    )
}

records_by_root = {
    accession_root: group.reset_index(drop=True)
    for accession_root, group in assembly_metadata.groupby(
        "accession_root",
        sort=False,
    )
}


def first_valid_accession(values):
    for value in values:
        if accession_pattern.fullmatch(str(value)):
            return str(value)
    return ""


def select_preferred_record(records):
    if records is None or records.empty:
        return None

    current_mask = (
        records["assembly_status"]
        .str.lower()
        .eq("current")
    )

    if current_mask.any():
        return records.loc[current_mask].iloc[0]

    return records.iloc[0]


validation_rows = []

for manifest_row in model_c_manifest.itertuples(
    index=False
):
    reported_accession = (
        manifest_row.reported_assembly_accession
    )

    accession_root = reported_accession.rsplit(
        ".",
        maxsplit=1,
    )[0]

    exact_records = records_by_accession.get(
        reported_accession
    )

    root_records = records_by_root.get(
        accession_root
    )

    if exact_records is not None:
        candidate_records = exact_records
    elif root_records is not None:
        candidate_records = root_records
    else:
        candidate_records = assembly_metadata.iloc[0:0]

    reported_record = select_preferred_record(
        exact_records
    )

    current_accession = ""

    if exact_records is not None:
        current_accession = first_valid_accession(
            exact_records["current_accession"]
        )

    if (
        not current_accession
        and not candidate_records.empty
    ):
        current_accession = first_valid_accession(
            candidate_records["current_accession"]
        )

    if (
        not current_accession
        and not candidate_records.empty
    ):
        current_records = candidate_records.loc[
            candidate_records["assembly_status"]
            .str.lower()
            .eq("current")
        ]

        current_accession = first_valid_accession(
            current_records["accession"]
        )

    if (
        not current_accession
        and reported_record is not None
    ):
        reported_status = str(
            reported_record["assembly_status"]
        ).lower()

        if not any(
            word in reported_status
            for word in [
                "suppressed",
                "replaced",
                "withdrawn",
            ]
        ):
            current_accession = reported_accession

    current_records = records_by_accession.get(
        current_accession
    )

    current_record = select_preferred_record(
        current_records
    )

    current_record_found = current_record is not None

    if reported_record is None:
        reported_status = ""
    else:
        reported_status = str(
            reported_record["assembly_status"]
        )

    if current_record_found:
        returned_biosample = str(
            current_record["biosample"]
        ).upper()

        organism_name = str(
            current_record["organism_name"]
        )

        organism_tax_id = str(
            current_record["organism_tax_id"]
        )

        current_status = str(
            current_record["assembly_status"]
        )

        suppression_reason = str(
            current_record["suppression_reason"]
        )

        assembly_level = str(
            current_record["assembly_level"]
        )

        source_database = str(
            current_record["source_database"]
        )

        paired_accession = str(
            current_record["paired_accession"]
        ).upper()

    else:
        returned_biosample = ""
        organism_name = ""
        organism_tax_id = ""
        current_status = ""
        suppression_reason = ""
        assembly_level = ""
        source_database = ""
        paired_accession = ""

    biosample_matches = (
        returned_biosample == manifest_row.biosample
    )

    ecoli_record = (
        organism_name.lower()
        .startswith("escherichia coli")
    )

    blocked_status = any(
        word in current_status.lower()
        for word in [
            "suppressed",
            "replaced",
            "withdrawn",
        ]
    )

    usable_for_download = bool(
        current_accession
        and current_record_found
        and biosample_matches
        and ecoli_record
        and not blocked_status
    )

    accession_changed = bool(
        current_accession
        and current_accession
        != reported_accession
    )

    if candidate_records.empty:
        validation_status = "no NCBI record"

    elif not current_accession:
        validation_status = "no current accession"

    elif not current_record_found:
        validation_status = (
            "current accession record not returned"
        )

    elif not biosample_matches:
        validation_status = "BioSample mismatch"

    elif not ecoli_record:
        validation_status = "non-E. coli record"

    elif blocked_status:
        validation_status = (
            "unavailable assembly status"
        )

    elif accession_changed:
        validation_status = (
            "updated accession confirmed"
        )

    else:
        validation_status = (
            "current accession confirmed"
        )

    validation_rows.append(
        {
            "model_c_row_index":
                manifest_row.model_c_row_index,

            "model_3b_row_index":
                manifest_row.model_3b_row_index,

            "biosample":
                manifest_row.biosample,

            "reported_assembly_accession":
                reported_accession,

            "reported_assembly_status":
                reported_status,

            "current_assembly_accession":
                current_accession,

            "current_assembly_status":
                current_status,

            "accession_changed":
                accession_changed,

            "returned_biosample":
                returned_biosample,

            "biosample_matches":
                biosample_matches,

            "organism_name":
                organism_name,

            "organism_tax_id":
                organism_tax_id,

            "assembly_level":
                assembly_level,

            "source_database":
                source_database,

            "paired_accession":
                paired_accession,

            "suppression_reason":
                suppression_reason,

            "validation_status":
                validation_status,

            "usable_for_download":
                usable_for_download,
        }
    )

accession_validation = pd.DataFrame(
    validation_rows
)

if len(accession_validation) != EXPECTED_MODEL_C_BIOSAMPLES:
    raise ValueError(
        f"Expected {EXPECTED_MODEL_C_BIOSAMPLES:,} validation "
        f"rows; found {len(accession_validation):,}."
    )

validation_path = (
    RESULTS_DIRECTORY
    / ACCESSION_VALIDATION_FILENAME
)

drive_validation_path = (
    DRIVE_NOTEBOOK23_DIRECTORY
    / ACCESSION_VALIDATION_FILENAME
)

accession_validation.to_csv(
    validation_path,
    index=False,
)

accession_validation.to_csv(
    drive_validation_path,
    index=False,
)

validation_summary = (
    accession_validation["validation_status"]
    .value_counts(dropna=False)
    .rename_axis("validation_status")
    .reset_index(name="pathogens")
)

display(validation_summary)

attention_rows = accession_validation.loc[
    ~accession_validation["usable_for_download"]
]

print(
    f"Usable for sequence download: "
    f"{accession_validation['usable_for_download'].sum():,}"
)

print(
    f"Require review: {len(attention_rows):,}"
)

if not attention_rows.empty:
    display(
        attention_rows[
            [
                "biosample",
                "reported_assembly_accession",
                "current_assembly_accession",
                "validation_status",
            ]
        ].head(20)
    )

print(f"Saved: {drive_validation_path}")

print(
    "\nTransition: The next cell will construct "
    "the targeted AMR-locus panel."
)

,validation_status,pathogens
0,current accession confirmed,9058


Usable for sequence download: 9,058
Require review: 0
Saved: /content/drive/MyDrive/Model3_MIC_Project/notebook23/23_current_assembly_validation.csv

Transition: The next cell will construct the targeted AMR-locus panel.


In [ ]:
#@title Cell 23.8 - Define the Model C targeted AMR locus panel
# This cell combines the 267 Model 3B full-gene labels, the 25 Model 3B
# point-target loci, and six additional Model C coding regions.

from pathlib import Path

import numpy as np
import pandas as pd


# Locate the Notebook 15B preprocessing file extracted by Cell 23.3.
search_directory = Path("/content/notebook23_work")

preprocessing_candidates = list(
    search_directory.rglob(
        "15B_pathogen_preprocessing.npz"
    )
)

if len(preprocessing_candidates) == 0:
    raise FileNotFoundError(
        "15B_pathogen_preprocessing.npz was not found under "
        "/content/notebook23_work. Rerun Cell 23.3."
    )

if len(preprocessing_candidates) > 1:
    raise ValueError(
        "More than one preprocessing file was found: "
        f"{preprocessing_candidates}"
    )

preprocessing_path = preprocessing_candidates[0]

print(f"Using: {preprocessing_path}")


# Load the Model 3B locus labels.
with np.load(
    preprocessing_path,
    allow_pickle=True,
) as preprocessing_archive:

    full_gene_columns = preprocessing_archive[
        "full_gene_columns"
    ].astype(str)

    point_target_columns = preprocessing_archive[
        "point_target_columns"
    ].astype(str)


full_gene_names = [
    name.removeprefix("full_gene__")
    for name in full_gene_columns
]

point_target_names = [
    name.removeprefix("point_target__")
    for name in point_target_columns
]

assert len(full_gene_names) == 267
assert len(point_target_names) == 25


# Identify the locus present in both Model 3B lists.
full_gene_set = set(full_gene_names)
point_target_set = set(point_target_names)

overlapping_loci = sorted(
    full_gene_set.intersection(point_target_set)
)

assert overlapping_loci == ["ampC"]


locus_rows = []


# Add the 267 Model 3B full-gene labels.
for locus_name in full_gene_names:

    if locus_name in point_target_set:
        locus_group = "AMR gene and point-target locus"
    else:
        locus_group = "AMR gene"

    locus_rows.append(
        {
            "locus_name": locus_name,
            "locus_group": locus_group,
            "sequence_region":
                "complete nucleotide sequence of the gene",
            "in_model_3b_full_gene_labels": True,
            "in_model_3b_point_target_loci":
                locus_name in point_target_set,
            "added_for_model_c": False,
        }
    )


# Add Model 3B point-target loci not already included above.
for locus_name in point_target_names:

    if locus_name in full_gene_set:
        continue

    if locus_name == "blaTEMp":
        locus_group = "AMR promoter-mutation region"
        sequence_region = (
            "AMRFinderPlus-defined promoter sequence"
        )
    else:
        locus_group = "AMR point-target locus"
        sequence_region = (
            "complete nucleotide sequence of the gene"
        )

    locus_rows.append(
        {
            "locus_name": locus_name,
            "locus_group": locus_group,
            "sequence_region": sequence_region,
            "in_model_3b_full_gene_labels": False,
            "in_model_3b_point_target_loci": True,
            "added_for_model_c": False,
        }
    )


# Add the six coding regions selected for Model C.
model_c_additions = [
    {
        "locus_name": "acrA",
        "locus_group": "efflux-pump component",
    },
    {
        "locus_name": "tolC",
        "locus_group":
            "efflux and outer-membrane channel",
    },
    {
        "locus_name": "marA",
        "locus_group": "efflux regulator",
    },
    {
        "locus_name": "rob",
        "locus_group": "efflux regulator",
    },
    {
        "locus_name": "ompR",
        "locus_group": "porin regulator",
    },
    {
        "locus_name": "envZ",
        "locus_group": "porin regulator",
    },
]


for locus in model_c_additions:
    locus_rows.append(
        {
            "locus_name": locus["locus_name"],
            "locus_group": locus["locus_group"],
            "sequence_region":
                "complete nucleotide sequence of the gene",
            "in_model_3b_full_gene_labels": False,
            "in_model_3b_point_target_loci": False,
            "added_for_model_c": True,
        }
    )


# Construct the final 297-locus panel.
target_locus_panel = pd.DataFrame(locus_rows)

required_cell9_columns = {
    "locus_name",
    "locus_group",
    "sequence_region",
    "in_model_3b_full_gene_labels",
    "in_model_3b_point_target_loci",
    "added_for_model_c",
}

missing_cell9_columns = (
    required_cell9_columns
    - set(target_locus_panel.columns)
)

if missing_cell9_columns:
    raise ValueError(
        "Columns required by Cell 23.9 are missing: "
        f"{sorted(missing_cell9_columns)}"
    )

assert target_locus_panel["locus_name"].is_unique
assert len(target_locus_panel) == 297

target_locus_panel.insert(
    0,
    "locus_id",
    [
        f"LOCUS_{number:04d}"
        for number in range(
            1,
            len(target_locus_panel) + 1,
        )
    ],
)

model_c_locus_panel = target_locus_panel.copy()


# Summarise the panel.
panel_summary = pd.DataFrame(
    [
        {
            "component": "Model 3B full-gene labels",
            "loci": len(full_gene_names),
        },
        {
            "component": "Model 3B point-target loci",
            "loci": len(point_target_names),
        },
        {
            "component": "Overlap counted once",
            "loci": len(overlapping_loci),
        },
        {
            "component": "Unique Model 3B loci",
            "loci": 291,
        },
        {
            "component": "Additional Model C regions",
            "loci": len(model_c_additions),
        },
        {
            "component": "Final Model C locus panel",
            "loci": len(target_locus_panel),
        },
    ]
)


# Save the corrected panel to Google Drive.
notebook23_directory = Path(
    "/content/drive/MyDrive/Model3_MIC_Project/notebook23"
)

notebook23_directory.mkdir(
    parents=True,
    exist_ok=True,
)

locus_panel_path = (
    notebook23_directory
    / "23_target_amr_locus_panel.csv"
)

target_locus_panel.to_csv(
    locus_panel_path,
    index=False,
)


display(panel_summary)

print(
    f"\nOverlapping Model 3B locus: "
    f"{overlapping_loci}"
)

display(
    target_locus_panel[
        target_locus_panel["added_for_model_c"]
    ][
        [
            "locus_id",
            "locus_name",
            "locus_group",
            "sequence_region",
        ]
    ]
)

print(f"Saved: {locus_panel_path}")

print(
    "\nTransition: Cell 23.9 will define the reference-sequence "
    "rule for each type of locus."
)

Using: /content/notebook23_work/notebook15B/15B_pathogen_preprocessing.npz


,component,loci
0,Model 3B full-gene labels,267
1,Model 3B point-target loci,25
2,Overlap counted once,1
3,Unique Model 3B loci,291
4,Additional Model C regions,6
5,Final Model C locus panel,297



Overlapping Model 3B locus: ['ampC']


,locus_id,locus_name,locus_group,sequence_region
291,LOCUS_0292,acrA,efflux-pump component,complete nucleotide sequence of the gene
292,LOCUS_0293,tolC,efflux and outer-membrane channel,complete nucleotide sequence of the gene
293,LOCUS_0294,marA,efflux regulator,complete nucleotide sequence of the gene
294,LOCUS_0295,rob,efflux regulator,complete nucleotide sequence of the gene
295,LOCUS_0296,ompR,porin regulator,complete nucleotide sequence of the gene
296,LOCUS_0297,envZ,porin regulator,complete nucleotide sequence of the gene


Saved: /content/drive/MyDrive/Model3_MIC_Project/notebook23/23_target_amr_locus_panel.csv

Transition: Cell 23.9 will define the reference-sequence rule for each type of locus.


In [ ]:
#@title Cell 23.9 - Define and validate Model C reference sequences
# This cell assigns the reference rule for all 297 Model C loci.
#
# 266 full-gene-only labels:
#   use the sequence extracted from each pathogen.
#
# 30 coding point-target or added loci:
#   use the E. coli K-12 MG1655 gene sequence.
#
# blaTEMp:
#   use the 100 nucleotides immediately upstream of blaTEM-1
#   in NCBI reference NG_050145.1.

from pathlib import Path

import re
import shutil
import subprocess
import sys
import urllib.request
import zipfile

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Load the corrected 297-locus panel
# ------------------------------------------------------------

notebook23_directory = Path(
    "/content/drive/MyDrive/Model3_MIC_Project/notebook23"
)

notebook23_directory.mkdir(
    parents=True,
    exist_ok=True,
)

locus_panel_path = (
    notebook23_directory
    / "23_target_amr_locus_panel.csv"
)

if "target_locus_panel" not in globals():

    if not locus_panel_path.exists():
        raise FileNotFoundError(
            "The corrected locus panel was not found. "
            "Rerun Cell 23.8."
        )

    target_locus_panel = pd.read_csv(
        locus_panel_path
    )


required_panel_columns = {
    "locus_id",
    "locus_name",
    "locus_group",
    "sequence_region",
    "in_model_3b_full_gene_labels",
    "in_model_3b_point_target_loci",
    "added_for_model_c",
}

missing_panel_columns = (
    required_panel_columns
    - set(target_locus_panel.columns)
)

if missing_panel_columns:
    raise ValueError(
        "Columns required from Cell 23.8 are missing: "
        f"{sorted(missing_panel_columns)}"
    )


boolean_columns = [
    "in_model_3b_full_gene_labels",
    "in_model_3b_point_target_loci",
    "added_for_model_c",
]

for column in boolean_columns:

    if target_locus_panel[column].dtype != bool:

        target_locus_panel[column] = (
            target_locus_panel[column]
            .astype(str)
            .str.lower()
            .map(
                {
                    "true": True,
                    "false": False,
                }
            )
        )

    if target_locus_panel[column].isna().any():
        raise ValueError(
            f"Invalid Boolean values in {column}."
        )


assert len(target_locus_panel) == 297
assert target_locus_panel["locus_name"].is_unique


# Define blaTEMp as the upstream sequence.
bla_temp_panel_mask = (
    target_locus_panel["locus_name"]
    == "blaTEMp"
)

assert int(bla_temp_panel_mask.sum()) == 1

target_locus_panel.loc[
    bla_temp_panel_mask,
    "sequence_region",
] = (
    "100 nucleotides immediately upstream of blaTEM"
)

target_locus_panel.to_csv(
    locus_panel_path,
    index=False,
)

model_c_locus_panel = (
    target_locus_panel.copy()
)


# ------------------------------------------------------------
# 2. Install Biopython if required
# ------------------------------------------------------------

try:
    from Bio import SeqIO

except ImportError:

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "biopython",
        ],
        check=True,
    )

    from Bio import SeqIO


# ------------------------------------------------------------
# 3. Locate the NCBI Datasets executable
# ------------------------------------------------------------

datasets_candidates = [
    Path("/content/notebook23_work/bin/datasets"),
]

system_datasets = shutil.which("datasets")

if system_datasets is not None:
    datasets_candidates.append(
        Path(system_datasets)
    )

datasets_executable = next(
    (
        path
        for path in datasets_candidates
        if path.exists()
    ),
    None,
)

if datasets_executable is None:
    raise FileNotFoundError(
        "The NCBI datasets executable was not found. "
        "Rerun Cell 23.5."
    )


# ------------------------------------------------------------
# 4. Locate or download the MG1655 GenBank record
# ------------------------------------------------------------

work_directory = Path(
    "/content/notebook23_work"
)

reference_directory = (
    work_directory / "reference_sequences"
)

reference_directory.mkdir(
    parents=True,
    exist_ok=True,
)

mg1655_accession = "GCF_000005845.2"

existing_gbff_candidates = [
    path
    for path in work_directory.rglob("*.gbff")
    if (
        mg1655_accession in str(path)
        or "000005845" in str(path)
    )
]

if existing_gbff_candidates:

    mg1655_gbff_path = (
        existing_gbff_candidates[0]
    )

else:

    mg1655_zip_path = (
        reference_directory
        / "GCF_000005845.2_MG1655.zip"
    )

    mg1655_extract_directory = (
        reference_directory
        / "GCF_000005845.2_MG1655"
    )

    if not zipfile.is_zipfile(
        mg1655_zip_path
    ):

        if mg1655_zip_path.exists():
            mg1655_zip_path.unlink()

        download_result = subprocess.run(
            [
                str(datasets_executable),
                "download",
                "genome",
                "accession",
                mg1655_accession,
                "--include",
                "gbff",
                "--filename",
                str(mg1655_zip_path),
            ],
            capture_output=True,
            text=True,
        )

        if download_result.returncode != 0:
            raise RuntimeError(
                "MG1655 download failed:\n"
                f"{download_result.stderr}"
            )

    if mg1655_extract_directory.exists():
        shutil.rmtree(
            mg1655_extract_directory
        )

    mg1655_extract_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    with zipfile.ZipFile(
        mg1655_zip_path,
        mode="r",
    ) as archive:
        archive.extractall(
            mg1655_extract_directory
        )

    extracted_gbff_candidates = list(
        mg1655_extract_directory.rglob(
            "*.gbff"
        )
    )

    if len(extracted_gbff_candidates) != 1:
        raise ValueError(
            "Expected one MG1655 GenBank file but found "
            f"{len(extracted_gbff_candidates)}."
        )

    mg1655_gbff_path = (
        extracted_gbff_candidates[0]
    )


print(
    f"MG1655 reference: {mg1655_gbff_path}"
)


# ------------------------------------------------------------
# 5. Index the MG1655 gene sequences
# ------------------------------------------------------------

def normalise_name(value):

    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower(),
    )


def split_qualifier_values(values):

    results = []

    for value in values:

        results.extend(
            part.strip()
            for part in re.split(
                r"[;,]",
                str(value),
            )
            if part.strip()
        )

    return results


primary_gene_index = {}
gene_alias_index = {}


for genbank_record in SeqIO.parse(
    str(mg1655_gbff_path),
    "genbank",
):

    for feature in genbank_record.features:

        if feature.type != "gene":
            continue

        primary_names = (
            feature.qualifiers.get(
                "gene",
                [],
            )
        )

        if not primary_names:
            continue

        gene_synonyms = split_qualifier_values(
            feature.qualifiers.get(
                "gene_synonym",
                [],
            )
        )

        locus_tags = (
            feature.qualifiers.get(
                "locus_tag",
                [],
            )
        )

        old_locus_tags = (
            feature.qualifiers.get(
                "old_locus_tag",
                [],
            )
        )

        sequence = str(
            feature.extract(
                genbank_record.seq
            )
        ).upper()

        locus_tag = (
            locus_tags[0]
            if locus_tags
            else ""
        )

        reference_record = {
            "primary_gene": primary_names[0],
            "locus_tag": locus_tag,
            "sequence": sequence,
            "length": len(sequence),
        }

        for primary_name in primary_names:

            primary_gene_index.setdefault(
                normalise_name(primary_name),
                [],
            ).append(reference_record)

        for alias in (
            gene_synonyms
            + locus_tags
            + old_locus_tags
        ):

            gene_alias_index.setdefault(
                normalise_name(alias),
                [],
            ).append(reference_record)


def remove_duplicate_reference_records(
    records
):

    unique_records = {}

    for record in records:

        key = (
            record["locus_tag"],
            record["sequence"],
        )

        unique_records[key] = record

    return list(
        unique_records.values()
    )


mg1655_manual_aliases = {
    "pmra": ["basR"],
    "pmrb": ["basS"],
}


def get_mg1655_reference(
    locus_name
):

    possible_names = [locus_name]

    possible_names.extend(
        mg1655_manual_aliases.get(
            normalise_name(locus_name),
            [],
        )
    )

    name_parts = re.split(
        r"[_-]",
        locus_name,
    )

    if len(name_parts) > 1:
        possible_names.append(
            name_parts[-1]
        )

    # Prefer an exact primary gene name.
    for possible_name in possible_names:

        candidates = (
            primary_gene_index.get(
                normalise_name(
                    possible_name
                ),
                [],
            )
        )

        candidates = (
            remove_duplicate_reference_records(
                candidates
            )
        )

        if len(candidates) == 1:
            return (
                "resolved",
                candidates[0],
                "",
            )

        if len(candidates) > 1:

            unique_sequences = {
                candidate["sequence"]
                for candidate in candidates
            }

            if len(unique_sequences) == 1:
                return (
                    "resolved",
                    candidates[0],
                    (
                        "Multiple annotations had "
                        "the same nucleotide sequence."
                    ),
                )

            return (
                "multiple MG1655 matches",
                None,
                (
                    "More than one different MG1655 "
                    "sequence matched the locus."
                ),
            )

    # Use an exact annotated synonym.
    alias_candidates = []

    for possible_name in possible_names:

        alias_candidates.extend(
            gene_alias_index.get(
                normalise_name(
                    possible_name
                ),
                [],
            )
        )

    alias_candidates = (
        remove_duplicate_reference_records(
            alias_candidates
        )
    )

    if len(alias_candidates) == 1:
        return (
            "resolved",
            alias_candidates[0],
            "Resolved through an MG1655 synonym.",
        )

    if len(alias_candidates) > 1:

        unique_sequences = {
            candidate["sequence"]
            for candidate in alias_candidates
        }

        if len(unique_sequences) == 1:
            return (
                "resolved",
                alias_candidates[0],
                (
                    "Multiple annotations had "
                    "the same nucleotide sequence."
                ),
            )

        return (
            "multiple MG1655 matches",
            None,
            (
                "More than one different MG1655 "
                "sequence matched a synonym."
            ),
        )

    return (
        "no MG1655 reference match",
        None,
        (
            "No exact MG1655 gene name or synonym "
            "matched this locus."
        ),
    )


# ------------------------------------------------------------
# 6. Download the blaTEM-1 reference record
# ------------------------------------------------------------

bla_temp_accession = "NG_050145.1"

bla_temp_genbank_path = (
    reference_directory
    / "NG_050145.1.gb"
)

if not bla_temp_genbank_path.exists():

    bla_temp_url = (
        "https://eutils.ncbi.nlm.nih.gov/entrez/"
        "eutils/efetch.fcgi"
        "?db=nuccore"
        "&id=NG_050145.1"
        "&rettype=gbwithparts"
        "&retmode=text"
    )

    request = urllib.request.Request(
        bla_temp_url,
        headers={
            "User-Agent":
                "Model-C-targeted-sequence-project"
        },
    )

    with urllib.request.urlopen(
        request,
        timeout=120,
    ) as response:

        genbank_content = (
            response.read()
        )

    with open(
        bla_temp_genbank_path,
        "wb",
    ) as output_file:

        output_file.write(
            genbank_content
        )


bla_temp_records = list(
    SeqIO.parse(
        str(bla_temp_genbank_path),
        "genbank",
    )
)

if len(bla_temp_records) != 1:
    raise ValueError(
        "Expected one NG_050145.1 GenBank record."
    )

bla_temp_record = (
    bla_temp_records[0]
)


# ------------------------------------------------------------
# 7. Extract the 100 nucleotides upstream of blaTEM-1
# ------------------------------------------------------------

bla_temp_cds_candidates = []


for feature in bla_temp_record.features:

    if feature.type != "CDS":
        continue

    feature_text = " ".join(
        value
        for values in (
            feature.qualifiers.values()
        )
        for value in values
    )

    normalised_feature_text = (
        normalise_name(feature_text)
    )

    if (
        "blatem"
        in normalised_feature_text
        or "tem1"
        in normalised_feature_text
    ):

        bla_temp_cds_candidates.append(
            feature
        )


if len(bla_temp_cds_candidates) != 1:
    raise ValueError(
        "Could not identify exactly one blaTEM-1 "
        "coding sequence in NG_050145.1."
    )


bla_temp_cds = (
    bla_temp_cds_candidates[0]
)

bla_temp_cds_start = int(
    bla_temp_cds.location.start
)

bla_temp_cds_end = int(
    bla_temp_cds.location.end
)

bla_temp_cds_strand = (
    bla_temp_cds.location.strand
)


# Biopython uses zero-based coordinates.
assert bla_temp_cds_start == 100
assert bla_temp_cds_end == 961


if bla_temp_cds_strand == 1:

    bla_temp_promoter_sequence = str(
        bla_temp_record.seq[
            bla_temp_cds_start - 100:
            bla_temp_cds_start
        ]
    ).upper()

    bla_temp_reference_coordinates = (
        "1-100"
    )

elif bla_temp_cds_strand == -1:

    bla_temp_promoter_sequence = str(
        bla_temp_record.seq[
            bla_temp_cds_end:
            bla_temp_cds_end + 100
        ]
        .reverse_complement()
    ).upper()

    bla_temp_reference_coordinates = (
        "962-1061 reverse-complemented"
    )

else:

    raise ValueError(
        "The blaTEM-1 coding-sequence direction "
        "was not defined."
    )


if len(bla_temp_promoter_sequence) != 100:
    raise ValueError(
        "The extracted blaTEM upstream reference "
        "is not 100 nucleotides long."
    )


print(
    "blaTEMp reference: "
    f"{bla_temp_accession}:"
    f"{bla_temp_reference_coordinates}"
)

print(
    "blaTEMp reference length: "
    f"{len(bla_temp_promoter_sequence)} nucleotides"
)


# ------------------------------------------------------------
# 8. Assign the reference rule to every locus
# ------------------------------------------------------------

reference_rows = []


for locus in target_locus_panel.itertuples(
    index=False
):

    locus_name = locus.locus_name

    is_full_gene = bool(
        locus.in_model_3b_full_gene_labels
    )

    is_point_target = bool(
        locus.in_model_3b_point_target_loci
    )

    is_model_c_addition = bool(
        locus.added_for_model_c
    )


    if locus_name == "blaTEMp":

        reference_rule = (
            "blaTEM-1 100-nucleotide "
            "upstream reference"
        )

        extraction_instruction = (
            "Locate blaTEM, determine its direction, "
            "extract the 100 nucleotides immediately "
            "upstream, and reverse-complement the "
            "sequence when blaTEM is on the reverse strand."
        )

        reference_accession = (
            f"{bla_temp_accession}:"
            f"{bla_temp_reference_coordinates}"
        )

        reference_sequence = (
            bla_temp_promoter_sequence
        )

        reference_length = len(
            bla_temp_promoter_sequence
        )

        reference_status = "resolved"

        reference_note = (
            "The reference is the 100-nucleotide "
            "sequence immediately upstream of the "
            "annotated blaTEM-1 coding sequence."
        )


    elif (
        is_point_target
        or is_model_c_addition
    ):

        reference_rule = (
            "E. coli K-12 MG1655 reference gene"
        )

        extraction_instruction = (
            "Locate the complete gene in the assembly "
            "and extract it in the gene direction."
        )

        (
            reference_status,
            reference_record,
            reference_note,
        ) = get_mg1655_reference(
            locus_name
        )

        if reference_record is not None:

            reference_accession = (
                f"{mg1655_accession}:"
                f"{reference_record['locus_tag']}"
            )

            reference_sequence = (
                reference_record["sequence"]
            )

            reference_length = (
                reference_record["length"]
            )

        else:

            reference_accession = ""
            reference_sequence = ""
            reference_length = np.nan


    elif is_full_gene:

        reference_rule = (
            "pathogen-specific AMRFinderPlus "
            "nucleotide sequence"
        )

        extraction_instruction = (
            "Use AMRFinderPlus to locate the gene "
            "and extract the nucleotide sequence "
            "detected in that pathogen."
        )

        reference_status = (
            "pathogen sequence will be extracted"
        )

        reference_accession = ""
        reference_sequence = ""
        reference_length = np.nan

        reference_note = (
            "No single fixed reference is assigned "
            "because this label may contain multiple "
            "valid nucleotide alleles."
        )


    else:

        raise ValueError(
            f"No reference rule was assigned to "
            f"{locus_name}."
        )


    review_required = (
        reference_status
        not in {
            "resolved",
            "pathogen sequence will be extracted",
        }
    )


    reference_rows.append(
        {
            "locus_id": locus.locus_id,
            "locus_name": locus_name,
            "locus_group": locus.locus_group,
            "sequence_region":
                locus.sequence_region,
            "reference_rule":
                reference_rule,
            "extraction_instruction":
                extraction_instruction,
            "reference_accession":
                reference_accession,
            "reference_length":
                reference_length,
            "reference_sequence":
                reference_sequence,
            "reference_status":
                reference_status,
            "reference_note":
                reference_note,
            "review_required":
                review_required,
        }
    )


reference_audit = pd.DataFrame(
    reference_rows
)


# ------------------------------------------------------------
# 9. Validate the three reference groups
# ------------------------------------------------------------

reference_rule_summary = pd.DataFrame(
    [
        {
            "reference_group":
                "Pathogen-specific full-gene sequence",
            "loci": int(
                (
                    reference_audit[
                        "reference_rule"
                    ]
                    ==
                    (
                        "pathogen-specific AMRFinderPlus "
                        "nucleotide sequence"
                    )
                ).sum()
            ),
        },
        {
            "reference_group":
                "MG1655 coding reference",
            "loci": int(
                (
                    reference_audit[
                        "reference_rule"
                    ]
                    ==
                    (
                        "E. coli K-12 MG1655 "
                        "reference gene"
                    )
                ).sum()
            ),
        },
        {
            "reference_group":
                "blaTEM-1 upstream reference",
            "loci": int(
                (
                    reference_audit[
                        "reference_rule"
                    ]
                    ==
                    (
                        "blaTEM-1 100-nucleotide "
                        "upstream reference"
                    )
                ).sum()
            ),
        },
    ]
)


assert (
    reference_rule_summary["loci"].tolist()
    == [266, 30, 1]
)

assert len(reference_audit) == 297
assert reference_audit["locus_id"].is_unique


# ------------------------------------------------------------
# 10. Save the audit and static reference sequences
# ------------------------------------------------------------

reference_audit_path = (
    notebook23_directory
    / "23_target_amr_reference_audit.csv"
)

reference_fasta_path = (
    notebook23_directory
    / "23_target_amr_reference_sequences.fasta"
)

reference_audit.to_csv(
    reference_audit_path,
    index=False,
)


resolved_static_references = (
    reference_audit[
        reference_audit["reference_status"]
        == "resolved"
    ]
)

references_requiring_review = (
    reference_audit[
        reference_audit["review_required"]
    ]
)


with open(
    reference_fasta_path,
    "w",
    encoding="utf-8",
) as fasta_file:

    for reference in (
        resolved_static_references.itertuples(
            index=False
        )
    ):

        header = (
            f">{reference.locus_id}"
            f"|{reference.locus_name}"
            f"|{reference.reference_accession}"
        )

        fasta_file.write(
            header + "\n"
        )

        sequence = (
            reference.reference_sequence
        )

        for start in range(
            0,
            len(sequence),
            80,
        ):

            fasta_file.write(
                sequence[
                    start:start + 80
                ]
                + "\n"
            )


# ------------------------------------------------------------
# 11. Display the result
# ------------------------------------------------------------

reference_status_summary = (
    reference_audit
    .groupby(
        "reference_status",
        as_index=False,
    )
    .agg(
        loci=("locus_id", "count")
    )
    .sort_values(
        "reference_status"
    )
    .reset_index(drop=True)
)


display(reference_rule_summary)

print()

display(reference_status_summary)

print(
    f"\nResolved static references: "
    f"{len(resolved_static_references)}"
)

print(
    f"References requiring review: "
    f"{len(references_requiring_review)}"
)


if len(references_requiring_review) > 0:

    display(
        references_requiring_review[
            [
                "locus_id",
                "locus_name",
                "locus_group",
                "reference_status",
                "reference_note",
            ]
        ]
    )

    print(
        "\nDo not begin sequence extraction until "
        "the displayed references have been reviewed."
    )

else:

    print(
        "\nAll required static reference sequences "
        "were resolved."
    )


print(f"\nSaved: {locus_panel_path}")
print(f"Saved: {reference_audit_path}")
print(f"Saved: {reference_fasta_path}")

if len(references_requiring_review) == 0:

    print(
        "\nTransition: Cell 23.10 will test targeted "
        "sequence extraction on a small assembly subset."
    )

MG1655 reference: /content/notebook23_work/reference_sequences/GCF_000005845.2_MG1655/ncbi_dataset/data/GCF_000005845.2/genomic.gbff
blaTEMp reference: NG_050145.1:1-100
blaTEMp reference length: 100 nucleotides


,reference_group,loci
0,Pathogen-specific full-gene sequence,266
1,MG1655 coding reference,30
2,blaTEM-1 upstream reference,1


,reference_status,loci
0,pathogen sequence will be extracted,266
1,resolved,31



Resolved static references: 31
References requiring review: 0

All required static reference sequences were resolved.

Saved: /content/drive/MyDrive/Model3_MIC_Project/notebook23/23_target_amr_locus_panel.csv
Saved: /content/drive/MyDrive/Model3_MIC_Project/notebook23/23_target_amr_reference_audit.csv
Saved: /content/drive/MyDrive/Model3_MIC_Project/notebook23/23_target_amr_reference_sequences.fasta

Transition: Cell 23.10 will test targeted sequence extraction on a small assembly subset.


In [ ]:
#@title Cell 23.10 - Create and validate the pilot extraction
# This portable cell creates every pilot file from the Model C manifest and
# the reference audit produced by the preceding cells. It then validates
# direct BLASTN extraction of the 30 MG1655 coding-reference loci.

from pathlib import Path
import importlib.util
import os
import platform
import re
import shutil
import subprocess
import sys
import tarfile
import time
import urllib.request
import zipfile

if importlib.util.find_spec("Bio") is None:
    print("Installing Biopython...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "biopython>=1.83,<2",
        ],
        check=True,
    )

import numpy as np
import pandas as pd
from Bio import SeqIO


PILOT_ASSEMBLY_COUNT = 3
MODEL_C_LOCUS_COUNT = 297
EXPECTED_PILOT_AMRFINDER_ROWS = 115
# AMRFinderPlus 4.2.7 reports 39 non-point AMR gene rows for the three
# pilot assemblies. Stress and virulence rows are outside the Model 3B
# full-gene panel, while point-mutation rows are handled by the 30 direct
# coding-locus searches below.
EXPECTED_PILOT_FULL_GENE_SEQUENCES = 39
MINIMUM_QUERY_COVERAGE = 90.0
MINIMUM_NUCLEOTIDE_IDENTITY = 90.0

# This pilot hit was reviewed after the first threshold-based run. It has
# 100% reference coverage, an intact reading frame with an in-frame
# nine-nucleotide deletion, no ambiguous nucleotides, no contig-boundary
# contact, and no overlap with the accepted ompF sequence.
REVIEWED_CODING_HITS = {
    ("GCA_000522325.1", "LOCUS_0281"): {
        "expected_contig": "KI929782.1",
        "minimum_identity": 89.0,
        "minimum_coverage": 99.0,
        "review_reason": (
            "reviewed full-length ompC hit; distinct from ompF"
        ),
    }
}

work_directory = Path("/content/notebook23_work")
notebook23_directory = Path(
    "/content/drive/MyDrive/Model3_MIC_Project/notebook23"
)
pilot_directory = notebook23_directory / "pilot_extraction"
pilot_work_directory = work_directory / "pilot_extraction"

pilot_directory.mkdir(parents=True, exist_ok=True)
pilot_work_directory.mkdir(parents=True, exist_ok=True)

pilot_manifest_path = (
    pilot_directory / "23_pilot_assembly_manifest.csv"
)
reference_audit_path = (
    notebook23_directory / "23_target_amr_reference_audit.csv"
)
sequence_details_path = (
    pilot_directory / "23_pilot_target_sequences.csv"
)
sequence_summary_path = (
    pilot_directory / "23_pilot_target_sequence_summary.csv"
)
amrfinder_results_path = (
    pilot_directory / "23_pilot_amrfinder_results.csv"
)
sequence_fasta_path = (
    pilot_directory / "23_pilot_target_sequences.fasta"
)
blast_results_path = (
    pilot_directory
    / "23_pilot_coding_reference_blast_results.csv"
)

required_input_files = [
    pilot_manifest_path,
    reference_audit_path,
    sequence_details_path,
    sequence_summary_path,
    amrfinder_results_path,
]

missing_input_files = [
    path for path in required_input_files if not path.exists()
]

if not reference_audit_path.exists():
    raise FileNotFoundError(
        f"Reference audit was not found: {reference_audit_path}. "
        "Run Cell 23.9, which defines and validates the reference "
        "sequences, before running this portable Cell 23.10."
    )


def ensure_amrfinderplus():
    environment_directory = (
        work_directory / "amrfinder_environment"
    )
    amrfinder_path = (
        environment_directory / "bin" / "amrfinder"
    )

    if not amrfinder_path.exists():
        print("Installing AMRFinderPlus 4.2.7...")
        bin_directory = work_directory / "bin"
        bin_directory.mkdir(parents=True, exist_ok=True)
        micromamba_path = bin_directory / "micromamba"

        if not micromamba_path.exists():
            machine = platform.machine().lower()
            if machine in {"x86_64", "amd64"}:
                micromamba_platform = "linux-64"
            elif machine in {"aarch64", "arm64"}:
                micromamba_platform = "linux-aarch64"
            else:
                raise RuntimeError(
                    "Unsupported Colab processor architecture: "
                    f"{machine}"
                )

            micromamba_url = (
                "https://micro.mamba.pm/api/micromamba/"
                f"{micromamba_platform}/latest"
            )
            micromamba_archive_path = (
                work_directory / "micromamba.tar.bz2"
            )
            with urllib.request.urlopen(
                micromamba_url,
                timeout=180,
            ) as response:
                with open(
                    micromamba_archive_path,
                    "wb",
                ) as archive_file:
                    shutil.copyfileobj(
                        response,
                        archive_file,
                    )

            with tarfile.open(
                micromamba_archive_path,
                "r:bz2",
            ) as archive:
                executable_member = archive.getmember(
                    "bin/micromamba"
                )
                extracted_executable = archive.extractfile(
                    executable_member
                )
                if extracted_executable is None:
                    raise RuntimeError(
                        "micromamba executable could not be "
                        "extracted."
                    )
                with open(
                    micromamba_path,
                    "wb",
                ) as executable_file:
                    shutil.copyfileobj(
                        extracted_executable,
                        executable_file,
                    )

            micromamba_path.chmod(0o755)
            micromamba_archive_path.unlink(
                missing_ok=True
            )

        micromamba_environment = os.environ.copy()
        micromamba_environment["MAMBA_ROOT_PREFIX"] = str(
            work_directory / "micromamba_root"
        )
        subprocess.run(
            [
                str(micromamba_path),
                "create",
                "--yes",
                "--prefix",
                str(environment_directory),
                "--override-channels",
                "--channel-priority",
                "strict",
                "--channel",
                "conda-forge",
                "--channel",
                "bioconda",
                "ncbi-amrfinderplus=4.2.7",
            ],
            check=True,
            env=micromamba_environment,
        )

    amrfinder_environment = os.environ.copy()
    amrfinder_environment["PATH"] = (
        str(amrfinder_path.parent)
        + os.pathsep
        + amrfinder_environment.get("PATH", "")
    )

    database_check = subprocess.run(
        [str(amrfinder_path), "--database_version"],
        capture_output=True,
        text=True,
        env=amrfinder_environment,
    )
    if database_check.returncode != 0:
        print("Downloading the AMRFinderPlus database...")
        subprocess.run(
            [str(amrfinder_path), "-u"],
            check=True,
            env=amrfinder_environment,
        )

    return amrfinder_path


portable_amrfinder_executable = ensure_amrfinderplus()


def download_genome_archive(
    datasets_path,
    assembly_accession,
    destination_path,
    command_environment=None,
):
    retry_delays = [0, 5, 15, 30]
    last_error = ""

    for attempt_number, retry_delay in enumerate(
        retry_delays,
        start=1,
    ):
        if retry_delay:
            print(
                f"NCBI download retry {attempt_number}/"
                f"{len(retry_delays)} for "
                f"{assembly_accession} after "
                f"{retry_delay} seconds."
            )
            time.sleep(retry_delay)

        destination_path.unlink(missing_ok=True)
        download_result = subprocess.run(
            [
                str(datasets_path),
                "download",
                "genome",
                "accession",
                assembly_accession,
                "--include",
                "genome",
                "--filename",
                str(destination_path),
            ],
            capture_output=True,
            text=True,
            env=command_environment,
        )

        if (
            download_result.returncode == 0
            and zipfile.is_zipfile(destination_path)
        ):
            return

        last_error = (
            download_result.stderr.strip()
            or download_result.stdout.strip()
            or "Downloaded file was not a valid ZIP archive."
        )

    destination_path.unlink(missing_ok=True)
    raise RuntimeError(
        f"NCBI download failed after "
        f"{len(retry_delays)} attempts for "
        f"{assembly_accession}: {last_error}"
    )


# ---------------------------------------------------------------------------
# Create the pilot inputs when this notebook is run on a new Google Drive.
# ---------------------------------------------------------------------------

pilot_files_to_create = [
    pilot_manifest_path,
    sequence_details_path,
    sequence_summary_path,
    amrfinder_results_path,
]

rebuild_pilot_files = any(
    not path.exists()
    for path in pilot_files_to_create
)

if not rebuild_pilot_files:
    existing_sequence_details = pd.read_csv(
        sequence_details_path,
        dtype=str,
    ).fillna("")
    if "sequence_source" in existing_sequence_details.columns:
        existing_full_gene_count = int(
            (
                existing_sequence_details["sequence_source"]
                == "AMRFinderPlus nucleotide hit"
            ).sum()
        )
    else:
        existing_full_gene_count = -1

    existing_amrfinder_results = pd.read_csv(
        amrfinder_results_path,
        dtype=str,
    )
    current_amrfinder_format = (
        "element_symbol"
        in existing_amrfinder_results.columns
    )

    if (
        existing_full_gene_count
        != EXPECTED_PILOT_FULL_GENE_SEQUENCES
        or not current_amrfinder_format
    ):
        rebuild_pilot_files = True
        print(
            "Earlier pilot files were detected. "
            "Rebuilding them with the current extraction rules."
        )

if rebuild_pilot_files:
    print(
        "Creating pilot files from the beginning."
    )

    reference_audit_bootstrap = pd.read_csv(
        reference_audit_path,
        dtype=str,
    )
    if len(reference_audit_bootstrap) != MODEL_C_LOCUS_COUNT:
        raise ValueError(
            "The reference audit does not contain 297 loci."
        )

    full_gene_rule_bootstrap = (
        "pathogen-specific AMRFinderPlus nucleotide sequence"
    )
    coding_reference_rule_bootstrap = (
        "E. coli K-12 MG1655 reference gene"
    )
    full_gene_panel_bootstrap = (
        reference_audit_bootstrap[
            reference_audit_bootstrap["reference_rule"]
            == full_gene_rule_bootstrap
        ].copy()
    )
    bla_temp_panel_bootstrap = (
        reference_audit_bootstrap[
            reference_audit_bootstrap["locus_name"]
            == "blaTEMp"
        ].copy()
    )

    assert len(full_gene_panel_bootstrap) == 266
    assert len(bla_temp_panel_bootstrap) == 1

    model_c_manifest_candidates = [
        work_directory
        / "results"
        / "23_model_c_assembly_manifest.csv",
        notebook23_directory
        / "23_model_c_assembly_manifest.csv",
    ]
    model_c_manifest_path = next(
        (
            path
            for path in model_c_manifest_candidates
            if path.exists()
        ),
        None,
    )
    if model_c_manifest_path is None:
        raise FileNotFoundError(
            "The Model C assembly manifest was not found. "
            "Rerun the manifest-creation cell."
        )

    model_c_manifest = pd.read_csv(
        model_c_manifest_path,
        dtype=str,
    )
    if (
        "assembly_accession"
        not in model_c_manifest.columns
        and "reported_assembly_accession"
        in model_c_manifest.columns
    ):
        model_c_manifest = model_c_manifest.rename(
            columns={
                "reported_assembly_accession":
                    "assembly_accession"
            }
        )

    required_manifest_columns = {
        "model_c_row_index",
        "model_3b_row_index",
        "biosample",
        "assembly_accession",
    }
    missing_manifest_columns = (
        required_manifest_columns
        - set(model_c_manifest.columns)
    )
    if missing_manifest_columns:
        raise ValueError(
            "Model C manifest columns are missing: "
            f"{sorted(missing_manifest_columns)}"
        )

    model_c_manifest["model_c_row_index"] = pd.to_numeric(
        model_c_manifest["model_c_row_index"],
        errors="raise",
    )
    pilot_manifest_bootstrap = (
        model_c_manifest.sort_values(
            "model_c_row_index"
        )
        .head(PILOT_ASSEMBLY_COUNT)
        .copy()
    )
    if len(pilot_manifest_bootstrap) != PILOT_ASSEMBLY_COUNT:
        raise ValueError(
            "Three pilot assemblies could not be selected."
        )
    pilot_manifest_bootstrap.to_csv(
        pilot_manifest_path,
        index=False,
    )

    def bootstrap_locate_executable(name, candidates):
        executable_candidates = [
            Path(path) for path in candidates
        ]
        system_path = shutil.which(name)
        if system_path:
            executable_candidates.append(
                Path(system_path)
            )

        executable = next(
            (
                path
                for path in executable_candidates
                if path.exists()
            ),
            None,
        )
        if executable is None:
            raise FileNotFoundError(
                f"{name} was not found. "
                "Rerun the software-installation cell."
            )
        return executable

    bootstrap_datasets_executable = (
        bootstrap_locate_executable(
            "datasets",
            [work_directory / "bin" / "datasets"],
        )
    )
    bootstrap_amrfinder_executable = (
        bootstrap_locate_executable(
            "amrfinder",
            [
                portable_amrfinder_executable
            ],
        )
    )

    bootstrap_environment = os.environ.copy()
    bootstrap_environment["PATH"] = (
        str(bootstrap_amrfinder_executable.parent)
        + os.pathsep
        + bootstrap_environment.get("PATH", "")
    )

    amrfinder_version_result = subprocess.run(
        [
            str(bootstrap_amrfinder_executable),
            "--version",
        ],
        capture_output=True,
        text=True,
        check=True,
        env=bootstrap_environment,
    )
    bootstrap_amrfinder_version = (
        amrfinder_version_result.stdout.strip()
        or amrfinder_version_result.stderr.strip()
    )
    print(
        f"AMRFinderPlus: "
        f"{bootstrap_amrfinder_version}"
    )

    database_version_result = subprocess.run(
        [
            str(bootstrap_amrfinder_executable),
            "--database_version",
        ],
        capture_output=True,
        text=True,
        check=True,
        env=bootstrap_environment,
    )
    bootstrap_database_version = (
        database_version_result.stdout.strip()
        or database_version_result.stderr.strip()
    )
    print(
        "AMRFinderPlus database is ready: "
        f"{bootstrap_database_version}"
    )

    def bootstrap_reverse_complement(sequence):
        table = str.maketrans(
            "ACGTRYKMSWBDHVNacgtrykmswbdhvn",
            "TGCAYRMKSWVHDBNtgcayrmkswvhdbn",
        )
        return sequence.translate(table)[::-1]

    def bootstrap_clean_column(column_name):
        return re.sub(
            r"[^a-z0-9]+",
            "_",
            str(column_name).strip().lower(),
        ).strip("_")

    def bootstrap_find_column(
        table,
        aliases,
        required=True,
    ):
        normalized_lookup = {
            bootstrap_clean_column(column): column
            for column in table.columns
        }
        for alias in aliases:
            normalized_alias = bootstrap_clean_column(
                alias
            )
            if normalized_alias in normalized_lookup:
                return normalized_lookup[
                    normalized_alias
                ]

        if required:
            raise ValueError(
                "Required AMRFinderPlus column was not found. "
                f"Expected one of {aliases}; observed "
                f"{list(table.columns)}"
            )
        return None

    def bootstrap_resolve_contig(
        reported_contig,
        assembly_sequences,
    ):
        reported = str(reported_contig).strip()
        if reported in assembly_sequences:
            return reported

        reported_variants = {
            reported,
            reported.removeprefix("lcl|"),
            reported.split()[0],
        }
        for contig_id in assembly_sequences:
            contig_variants = {
                contig_id,
                contig_id.removeprefix("lcl|"),
                contig_id.split()[0],
            }
            if reported_variants & contig_variants:
                return contig_id

        raise KeyError(
            f"Contig {reported_contig} was not found "
            "in the assembly FASTA."
        )

    def bootstrap_extract_interval(
        assembly_sequences,
        contig_id,
        start,
        stop,
        strand,
    ):
        resolved_contig = bootstrap_resolve_contig(
            contig_id,
            assembly_sequences,
        )
        contig_sequence = assembly_sequences[
            resolved_contig
        ]
        left = min(int(start), int(stop))
        right = max(int(start), int(stop))

        if left < 1 or right > len(contig_sequence):
            raise ValueError(
                "Reported sequence coordinates fall outside "
                "the assembly contig."
            )

        sequence = contig_sequence[left - 1:right]
        normalized_strand = str(strand).strip()
        if normalized_strand == "-" or int(start) > int(stop):
            sequence = bootstrap_reverse_complement(
                sequence
            )
            normalized_strand = "-"
        else:
            normalized_strand = "+"

        return {
            "contig_id": resolved_contig,
            "start": left,
            "stop": right,
            "strand": normalized_strand,
            "sequence": sequence,
            "sequence_length": len(sequence),
            "touches_contig_edge": (
                left == 1
                or right == len(contig_sequence)
            ),
        }

    def bootstrap_label_base(label):
        return (
            str(label)
            .split("=", 1)[0]
            .strip()
            .casefold()
        )

    full_gene_records_bootstrap = (
        full_gene_panel_bootstrap.to_dict(
            orient="records"
        )
    )
    full_gene_exact_bootstrap = {}
    full_gene_base_bootstrap = {}
    for record in full_gene_records_bootstrap:
        exact_key = (
            str(record["locus_name"])
            .strip()
            .casefold()
        )
        base_key = bootstrap_label_base(
            record["locus_name"]
        )
        full_gene_exact_bootstrap.setdefault(
            exact_key,
            [],
        ).append(record)
        full_gene_base_bootstrap.setdefault(
            base_key,
            [],
        ).append(record)

    def bootstrap_choose_full_gene_locus(
        gene_symbol,
        report_row,
    ):
        symbol = str(gene_symbol).strip()
        exact_key = symbol.casefold()
        base_key = bootstrap_label_base(symbol)

        report_text = " ".join(
            str(value)
            for value in report_row.values
            if not pd.isna(value)
        ).upper()

        qualifier = None
        if (
            "MISTRANSLATION" in report_text
            or "FRAME_SHIFT" in report_text
            or "FRAMESHIFT" in report_text
        ):
            qualifier = "MISTRANSLATION"
        elif "PARTIAL" in report_text:
            qualifier = "PARTIAL"

        base_candidates = (
            full_gene_base_bootstrap.get(
                base_key,
                [],
            )
        )

        if qualifier is not None:
            qualified = [
                record
                for record in base_candidates
                if str(
                    record["locus_name"]
                ).upper().endswith(
                    f"={qualifier}"
                )
            ]
            if len(qualified) == 1:
                return qualified[0], (
                    f"matched {qualifier} "
                    "AMRFinderPlus call"
                )

        exact_candidates = (
            full_gene_exact_bootstrap.get(
                exact_key,
                [],
            )
        )
        if len(exact_candidates) == 1:
            return (
                exact_candidates[0],
                "exact gene-symbol match",
            )

        unsuffixed = [
            record
            for record in base_candidates
            if "=" not in str(record["locus_name"])
        ]
        if len(unsuffixed) == 1:
            return (
                unsuffixed[0],
                "base gene-symbol match",
            )

        if len(base_candidates) == 1:
            return (
                base_candidates[0],
                "unique base-label match",
            )

        if len(base_candidates) == 0:
            return (
                None,
                "AMR gene is not represented "
                "in the 266-locus panel",
            )

        return (
            None,
            "AMR gene maps to multiple Model 3B labels",
        )

    bootstrap_sequence_rows = []
    bootstrap_amrfinder_tables = []

    for pilot_row in pilot_manifest_bootstrap.itertuples(
        index=False
    ):
        biosample = pilot_row.biosample
        assembly_accession = (
            pilot_row.assembly_accession
        )
        print(
            f"Pilot AMRFinderPlus extraction: "
            f"{assembly_accession}"
        )

        assembly_directory = (
            pilot_work_directory
            / assembly_accession
        )
        assembly_directory.mkdir(
            parents=True,
            exist_ok=True,
        )
        assembly_archive_path = (
            assembly_directory
            / f"{assembly_accession}.zip"
        )

        fasta_candidates = list(
            assembly_directory.rglob("*_genomic.fna")
        )
        if len(fasta_candidates) == 1:
            assembly_fasta_path = fasta_candidates[0]
        else:
            download_genome_archive(
                bootstrap_datasets_executable,
                assembly_accession,
                assembly_archive_path,
                bootstrap_environment,
            )

            extracted_directory = (
                assembly_directory
                / "bootstrap_extracted"
            )
            if extracted_directory.exists():
                shutil.rmtree(extracted_directory)
            extracted_directory.mkdir(
                parents=True,
                exist_ok=True,
            )
            with zipfile.ZipFile(
                assembly_archive_path,
                "r",
            ) as archive:
                archive.extractall(extracted_directory)

            fasta_candidates = list(
                extracted_directory.rglob(
                    "*_genomic.fna"
                )
            )
            if len(fasta_candidates) != 1:
                raise ValueError(
                    f"Expected one FASTA for "
                    f"{assembly_accession}; found "
                    f"{len(fasta_candidates)}."
                )
            assembly_fasta_path = fasta_candidates[0]

        assembly_sequences = {
            record.id: str(record.seq).upper()
            for record in SeqIO.parse(
                str(assembly_fasta_path),
                "fasta",
            )
        }
        if not assembly_sequences:
            raise ValueError(
                f"No sequences were read for "
                f"{assembly_accession}."
            )

        amrfinder_report_path = (
            assembly_directory / "amrfinder.tsv"
        )
        amrfinder_result = subprocess.run(
            [
                str(bootstrap_amrfinder_executable),
                "--nucleotide",
                str(assembly_fasta_path),
                "--organism",
                "Escherichia",
                "--plus",
                "--threads",
                "2",
                "--output",
                str(amrfinder_report_path),
            ],
            capture_output=True,
            text=True,
            env=bootstrap_environment,
        )
        if amrfinder_result.returncode != 0:
            raise RuntimeError(
                f"AMRFinderPlus failed for "
                f"{assembly_accession}: "
                f"{amrfinder_result.stderr}"
            )

        if (
            not amrfinder_report_path.exists()
            or amrfinder_report_path.stat().st_size == 0
        ):
            amrfinder_report = pd.DataFrame()
        else:
            amrfinder_report = pd.read_csv(
                amrfinder_report_path,
                sep="\t",
                dtype=str,
            )
            amrfinder_report.columns = [
                bootstrap_clean_column(column)
                for column in amrfinder_report.columns
            ]

        if amrfinder_report.empty:
            continue

        amrfinder_report.insert(
            0,
            "assembly_accession",
            assembly_accession,
        )
        amrfinder_report.insert(
            0,
            "biosample",
            biosample,
        )
        amrfinder_report.insert(
            2,
            "amrfinder_row_index",
            range(len(amrfinder_report)),
        )
        amrfinder_report["mapped_locus_id"] = ""
        amrfinder_report["mapped_locus_name"] = ""
        amrfinder_report["mapping_status"] = ""

        gene_column = bootstrap_find_column(
            amrfinder_report,
            [
                "gene_symbol",
                "genesymbol",
                "element_symbol",
                "elementsymbol",
            ],
        )
        contig_column = bootstrap_find_column(
            amrfinder_report,
            ["contig_id", "contig"],
        )
        start_column = bootstrap_find_column(
            amrfinder_report,
            ["start"],
        )
        stop_column = bootstrap_find_column(
            amrfinder_report,
            ["stop"],
        )
        strand_column = bootstrap_find_column(
            amrfinder_report,
            ["strand"],
        )
        type_column = bootstrap_find_column(
            amrfinder_report,
            ["element_type", "type"],
            required=False,
        )
        subtype_column = bootstrap_find_column(
            amrfinder_report,
            ["element_subtype", "subtype"],
            required=False,
        )
        method_column = bootstrap_find_column(
            amrfinder_report,
            ["method"],
            required=False,
        )
        identity_column = bootstrap_find_column(
            amrfinder_report,
            [
                "identity_to_reference",
                "identity_to_reference_sequence",
                "percent_identity_to_reference_sequence",
            ],
            required=False,
        )
        coverage_column = bootstrap_find_column(
            amrfinder_report,
            [
                "coverage_of_reference",
                "coverage_of_reference_sequence",
                "percent_coverage_of_reference_sequence",
            ],
            required=False,
        )

        seen_locus_hits = set()
        seen_bla_tem_hits = set()

        for report_index, report_row in (
            amrfinder_report.iterrows()
        ):
            gene_symbol_value = report_row[
                gene_column
            ]
            if (
                pd.isna(gene_symbol_value)
                or str(gene_symbol_value).strip()
                in {"", "NA"}
            ):
                continue
            gene_symbol = str(
                gene_symbol_value
            ).strip()

            element_type = (
                str(report_row[type_column]).upper()
                if type_column is not None
                else ""
            )
            element_subtype = (
                str(
                    report_row[subtype_column]
                ).upper()
                if subtype_column is not None
                else ""
            )
            if (
                type_column is not None
                and element_type != "AMR"
            ):
                amrfinder_report.loc[
                    report_index,
                    "mapping_status",
                ] = "not an AMR element"
                continue
            if "POINT" in element_subtype:
                amrfinder_report.loc[
                    report_index,
                    "mapping_status",
                ] = "point-mutation report row"
                continue

            coordinate_values = [
                report_row[contig_column],
                report_row[start_column],
                report_row[stop_column],
                report_row[strand_column],
            ]
            if any(
                pd.isna(value)
                or str(value).strip() in {"", "NA"}
                for value in coordinate_values
            ):
                amrfinder_report.loc[
                    report_index,
                    "mapping_status",
                ] = "coordinates unavailable"
                continue

            contig_id = bootstrap_resolve_contig(
                report_row[contig_column],
                assembly_sequences,
            )
            start = int(
                float(report_row[start_column])
            )
            stop = int(
                float(report_row[stop_column])
            )
            strand = str(
                report_row[strand_column]
            ).strip()

            locus_record, mapping_reason = (
                bootstrap_choose_full_gene_locus(
                    gene_symbol,
                    report_row,
                )
            )
            amrfinder_report.loc[
                report_index,
                "mapping_status",
            ] = mapping_reason

            physical_key = (
                contig_id,
                min(start, stop),
                max(start, stop),
                strand,
            )

            if locus_record is not None:
                amrfinder_report.loc[
                    report_index,
                    "mapped_locus_id",
                ] = locus_record["locus_id"]
                amrfinder_report.loc[
                    report_index,
                    "mapped_locus_name",
                ] = locus_record["locus_name"]

                locus_hit_key = (
                    locus_record["locus_id"],
                    physical_key,
                )
                if locus_hit_key not in seen_locus_hits:
                    seen_locus_hits.add(
                        locus_hit_key
                    )
                    extracted = (
                        bootstrap_extract_interval(
                            assembly_sequences,
                            contig_id,
                            start,
                            stop,
                            strand,
                        )
                    )
                    bootstrap_sequence_rows.append(
                        {
                            "biosample": biosample,
                            "assembly_accession":
                                assembly_accession,
                            "locus_id":
                                locus_record["locus_id"],
                            "locus_name":
                                locus_record["locus_name"],
                            "sequence_source":
                                "AMRFinderPlus nucleotide hit",
                            **extracted,
                            "detection_method": (
                                str(
                                    report_row[
                                        method_column
                                    ]
                                )
                                if method_column is not None
                                else ""
                            ),
                            "percent_identity": (
                                pd.to_numeric(
                                    report_row[
                                        identity_column
                                    ],
                                    errors="coerce",
                                )
                                if identity_column is not None
                                else np.nan
                            ),
                            "query_coverage": (
                                pd.to_numeric(
                                    report_row[
                                        coverage_column
                                    ],
                                    errors="coerce",
                                )
                                if coverage_column is not None
                                else np.nan
                            ),
                            "reference_length": np.nan,
                        }
                    )

            normalized_symbol = re.sub(
                r"[^a-z0-9]+",
                "",
                gene_symbol.casefold(),
            )
            if not normalized_symbol.startswith(
                "blatem"
            ):
                continue

            bla_tem_key = (
                contig_id,
                min(start, stop),
                max(start, stop),
                strand,
            )
            if bla_tem_key in seen_bla_tem_hits:
                continue
            seen_bla_tem_hits.add(bla_tem_key)

            contig_sequence = assembly_sequences[
                contig_id
            ]
            gene_left = min(start, stop)
            gene_right = max(start, stop)

            if strand == "-" or start > stop:
                promoter_start = gene_right + 1
                promoter_stop = gene_right + 100
                promoter_strand = "-"
            else:
                promoter_start = gene_left - 100
                promoter_stop = gene_left - 1
                promoter_strand = "+"

            if (
                promoter_start < 1
                or promoter_stop > len(contig_sequence)
            ):
                continue

            promoter = bootstrap_extract_interval(
                assembly_sequences,
                contig_id,
                promoter_start,
                promoter_stop,
                promoter_strand,
            )
            bootstrap_sequence_rows.append(
                {
                    "biosample": biosample,
                    "assembly_accession":
                        assembly_accession,
                    "locus_id":
                        bla_temp_panel_bootstrap.iloc[
                            0
                        ]["locus_id"],
                    "locus_name": "blaTEMp",
                    "sequence_source": (
                        "100 nucleotides immediately upstream "
                        "of an AMRFinderPlus blaTEM hit"
                    ),
                    **promoter,
                    "detection_method":
                        "blaTEM-oriented upstream extraction",
                    "percent_identity": np.nan,
                    "query_coverage": 100.0,
                    "reference_length": 100,
                }
            )

        bootstrap_amrfinder_tables.append(
            amrfinder_report
        )

    bootstrap_sequence_details = pd.DataFrame(
        bootstrap_sequence_rows
    )
    if bootstrap_sequence_details.empty:
        raise ValueError(
            "No pilot AMR sequences were extracted."
        )

    bootstrap_sequence_details = (
        bootstrap_sequence_details.sort_values(
            [
                "assembly_accession",
                "locus_id",
                "contig_id",
                "start",
            ]
        ).reset_index(drop=True)
    )
    bootstrap_sequence_details["copy_index"] = (
        bootstrap_sequence_details.groupby(
            ["assembly_accession", "locus_id"]
        ).cumcount()
        + 1
    )

    bootstrap_detail_counts = (
        bootstrap_sequence_details.groupby(
            ["assembly_accession", "locus_id"]
        )
        .size()
        .to_dict()
    )
    bootstrap_summary_rows = []
    for pilot_row in pilot_manifest_bootstrap.itertuples(
        index=False
    ):
        for locus in (
            reference_audit_bootstrap.itertuples(
                index=False
            )
        ):
            key = (
                pilot_row.assembly_accession,
                locus.locus_id,
            )
            copy_number = int(
                bootstrap_detail_counts.get(key, 0)
            )

            if copy_number > 0:
                extraction_status = "sequence extracted"
            elif locus.locus_name == "blaTEMp":
                extraction_status = "blaTEM not detected"
            elif (
                locus.reference_rule
                == full_gene_rule_bootstrap
            ):
                extraction_status = (
                    "not detected by AMRFinderPlus"
                )
            elif (
                locus.reference_rule
                == coding_reference_rule_bootstrap
            ):
                extraction_status = (
                    "awaiting direct BLASTN extraction"
                )
            else:
                extraction_status = "not extracted"

            bootstrap_summary_rows.append(
                {
                    "biosample": pilot_row.biosample,
                    "assembly_accession":
                        pilot_row.assembly_accession,
                    "locus_id": locus.locus_id,
                    "locus_name": locus.locus_name,
                    "reference_rule":
                        locus.reference_rule,
                    "extraction_status":
                        extraction_status,
                    "copy_number": copy_number,
                }
            )

    bootstrap_sequence_summary = pd.DataFrame(
        bootstrap_summary_rows
    )
    if len(bootstrap_sequence_summary) != (
        PILOT_ASSEMBLY_COUNT * MODEL_C_LOCUS_COUNT
    ):
        raise ValueError(
            "Pilot summary does not contain 891 rows."
        )

    bootstrap_amrfinder_results = pd.concat(
        bootstrap_amrfinder_tables,
        ignore_index=True,
        sort=False,
    )

    bootstrap_sequence_details.to_csv(
        sequence_details_path,
        index=False,
    )
    bootstrap_sequence_summary.to_csv(
        sequence_summary_path,
        index=False,
    )
    bootstrap_amrfinder_results.to_csv(
        amrfinder_results_path,
        index=False,
    )

    print(
        "Pilot manifest, AMRFinderPlus results, "
        "sequences, and summary were created."
    )


missing_input_files = [
    path for path in required_input_files if not path.exists()
]
if missing_input_files:
    raise FileNotFoundError(
        "Pilot bootstrap did not create all required files: "
        f"{missing_input_files}"
    )


# Load the successfully saved pilot results.
pilot_manifest = pd.read_csv(pilot_manifest_path, dtype=str)
reference_audit = pd.read_csv(reference_audit_path, dtype=str)
previous_sequence_details = pd.read_csv(
    sequence_details_path,
    dtype=str,
)
previous_sequence_summary = pd.read_csv(
    sequence_summary_path,
    dtype=str,
)
combined_amrfinder_results = pd.read_csv(
    amrfinder_results_path,
    dtype=str,
)

assert len(pilot_manifest) == PILOT_ASSEMBLY_COUNT
assert len(reference_audit) == MODEL_C_LOCUS_COUNT

full_gene_rule = (
    "pathogen-specific AMRFinderPlus nucleotide sequence"
)
coding_reference_rule = (
    "E. coli K-12 MG1655 reference gene"
)

full_gene_panel = reference_audit[
    reference_audit["reference_rule"] == full_gene_rule
].copy()
coding_reference_panel = reference_audit[
    reference_audit["reference_rule"] == coding_reference_rule
].copy()
bla_temp_panel = reference_audit[
    reference_audit["locus_name"] == "blaTEMp"
].copy()

assert len(full_gene_panel) == 266
assert len(coding_reference_panel) == 30
assert len(bla_temp_panel) == 1

if (
    coding_reference_panel["reference_sequence"].isna()
    | coding_reference_panel["reference_sequence"].eq("")
).any():
    raise ValueError(
        "One or more MG1655 coding references have no sequence."
    )


# Retain only the successful AMRFinderPlus and blaTEMp sequences.
retained_sequence_details = previous_sequence_details[
    (
        previous_sequence_details["sequence_source"]
        == "AMRFinderPlus nucleotide hit"
    )
    | (previous_sequence_details["locus_name"] == "blaTEMp")
].copy()

retained_sequence_details = retained_sequence_details.drop(
    columns=["copy_index"],
    errors="ignore",
)


# Locate NCBI Datasets and BLASTN.
datasets_candidates = [work_directory / "bin" / "datasets"]
system_datasets = shutil.which("datasets")
if system_datasets:
    datasets_candidates.append(Path(system_datasets))

datasets_executable = next(
    (path for path in datasets_candidates if path.exists()),
    None,
)
if datasets_executable is None:
    raise FileNotFoundError(
        "NCBI Datasets was not found. Rerun Cell 23.5."
    )

blastn_candidates = [
    work_directory / "amrfinder_environment" / "bin" / "blastn"
]
system_blastn = shutil.which("blastn")
if system_blastn:
    blastn_candidates.append(Path(system_blastn))

blastn_executable = next(
    (path for path in blastn_candidates if path.exists()),
    None,
)
if blastn_executable is None:
    raise FileNotFoundError(
        "BLASTN was not found in the current Colab runtime."
    )

blast_version = subprocess.run(
    [str(blastn_executable), "-version"],
    capture_output=True,
    text=True,
    check=True,
).stdout.splitlines()[0]
print(blast_version)


# Write the 30 MG1655 reference sequences.
query_fasta_path = (
    pilot_work_directory / "23_coding_reference_queries.fasta"
)

with open(query_fasta_path, "w", encoding="utf-8") as fasta_file:
    for reference in coding_reference_panel.itertuples(index=False):
        fasta_file.write(f">{reference.locus_id}\n")
        sequence = str(reference.reference_sequence).upper()
        for start in range(0, len(sequence), 80):
            fasta_file.write(sequence[start:start + 80] + "\n")

coding_reference_lookup = (
    coding_reference_panel.set_index("locus_id").to_dict(
        orient="index"
    )
)


def reverse_complement(sequence):
    table = str.maketrans(
        "ACGTRYKMSWBDHVNacgtrykmswbdhvn",
        "TGCAYRMKSWVHDBNtgcayrmkswvhdbn",
    )
    return sequence.translate(table)[::-1]


STOP_CODONS = {"TAA", "TAG", "TGA"}
BACTERIAL_START_CODONS = {
    "ATG",
    "GTG",
    "TTG",
    "CTG",
    "ATT",
    "ATC",
    "ATA",
}


def extract_reference_oriented_sequence(hit, assembly_sequences):
    contig_sequence = assembly_sequences[hit.contig_id]

    query_start = int(hit.query_start)
    query_stop = int(hit.query_stop)
    reference_length = int(hit.reference_length)
    subject_start = int(hit.subject_start)
    subject_stop = int(hit.subject_stop)

    if subject_start <= subject_stop:
        strand = "+"
        extraction_start = subject_start - (query_start - 1)
        extraction_stop = (
            subject_stop + (reference_length - query_stop)
        )
    else:
        strand = "-"
        extraction_start = (
            subject_stop - (reference_length - query_stop)
        )
        extraction_stop = subject_start + (query_start - 1)

    unclipped_start = extraction_start
    unclipped_stop = extraction_stop
    extraction_start = max(1, extraction_start)
    extraction_stop = min(
        len(contig_sequence),
        extraction_stop,
    )
    touches_contig_edge = (
        unclipped_start < 1
        or unclipped_stop > len(contig_sequence)
        or extraction_start == 1
        or extraction_stop == len(contig_sequence)
    )

    extracted_sequence = contig_sequence[
        extraction_start - 1:extraction_stop
    ]
    if strand == "-":
        extracted_sequence = reverse_complement(
            extracted_sequence
        )

    return {
        "contig_id": hit.contig_id,
        "contig_length": len(contig_sequence),
        "start": extraction_start,
        "stop": extraction_stop,
        "strand": strand,
        "sequence": extracted_sequence,
        "sequence_length": len(extracted_sequence),
        "touches_contig_edge": touches_contig_edge,
    }


def assess_coding_integrity(
    sequence,
    reference_sequence,
    touches_contig_edge,
):
    sequence = str(sequence).upper()
    reference_sequence = str(reference_sequence).upper()

    ambiguous_nucleotides = sum(
        nucleotide not in {"A", "C", "G", "T"}
        for nucleotide in sequence
    )
    length_difference = (
        len(sequence) - len(reference_sequence)
    )
    length_is_multiple_of_three = len(sequence) % 3 == 0

    complete_codons = [
        sequence[position:position + 3]
        for position in range(0, len(sequence) - 2, 3)
    ]
    start_codon = (
        complete_codons[0] if complete_codons else ""
    )
    terminal_codon = (
        complete_codons[-1] if complete_codons else ""
    )
    internal_stop_count = sum(
        codon in STOP_CODONS
        for codon in complete_codons[:-1]
    )

    reference_has_terminal_stop = (
        len(reference_sequence) >= 3
        and reference_sequence[-3:] in STOP_CODONS
    )
    recognised_start = (
        start_codon in BACTERIAL_START_CODONS
    )
    terminal_stop_present = (
        terminal_codon in STOP_CODONS
    )

    uncertainty_reasons = []
    disruption_reasons = []

    if touches_contig_edge:
        uncertainty_reasons.append(
            "sequence reaches a contig boundary"
        )
    if ambiguous_nucleotides > 0:
        uncertainty_reasons.append(
            f"{ambiguous_nucleotides} ambiguous nucleotide(s)"
        )

    if not length_is_multiple_of_three:
        disruption_reasons.append(
            "length is not a multiple of three"
        )
    if internal_stop_count > 0:
        disruption_reasons.append(
            f"{internal_stop_count} internal stop codon(s)"
        )
    if not recognised_start:
        disruption_reasons.append(
            f"unrecognised start codon {start_codon or 'none'}"
        )
    if (
        reference_has_terminal_stop
        and not terminal_stop_present
    ):
        disruption_reasons.append(
            "terminal stop codon is absent"
        )

    if uncertainty_reasons:
        coding_integrity = "uncertain"
        integrity_reason = "; ".join(
            uncertainty_reasons + disruption_reasons
        )
    elif disruption_reasons:
        coding_integrity = "potentially disrupted"
        integrity_reason = "; ".join(disruption_reasons)
    else:
        coding_integrity = "intact ORF"
        if length_difference == 0:
            integrity_reason = "no coding disruption detected"
        else:
            integrity_reason = (
                f"in-frame length difference of "
                f"{length_difference:+d} nucleotide(s)"
            )

    return {
        "coding_integrity": coding_integrity,
        "integrity_reason": integrity_reason,
        "length_difference": length_difference,
        "ambiguous_nucleotides": ambiguous_nucleotides,
        "start_codon": start_codon,
        "internal_stop_count": internal_stop_count,
        "terminal_codon": terminal_codon,
    }


def locate_assembly_fasta(assembly_accession):
    assembly_directory = (
        pilot_work_directory / assembly_accession
    )
    assembly_directory.mkdir(parents=True, exist_ok=True)

    existing = list(
        assembly_directory.rglob("*_genomic.fna")
    )
    if len(existing) == 1:
        return existing[0]

    assembly_zip_path = (
        assembly_directory / f"{assembly_accession}.zip"
    )

    if not zipfile.is_zipfile(assembly_zip_path):
        download_genome_archive(
            datasets_executable,
            assembly_accession,
            assembly_zip_path,
        )

    extracted_directory = (
        assembly_directory / "blast_extracted"
    )
    if extracted_directory.exists():
        shutil.rmtree(extracted_directory)
    extracted_directory.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(assembly_zip_path, "r") as archive:
        archive.extractall(extracted_directory)

    candidates = list(
        extracted_directory.rglob("*_genomic.fna")
    )
    if len(candidates) != 1:
        raise ValueError(
            f"Expected one FASTA for {assembly_accession}; "
            f"found {len(candidates)}."
        )
    return candidates[0]


blast_columns = [
    "locus_id",
    "contig_id",
    "percent_identity",
    "alignment_length",
    "reference_length",
    "query_start",
    "query_stop",
    "subject_start",
    "subject_stop",
    "bit_score",
    "e_value",
]

all_blast_tables = []
direct_sequence_rows = []
assembly_sequence_cache = {}


for pilot_row in pilot_manifest.itertuples(index=False):
    biosample = pilot_row.biosample
    assembly_accession = pilot_row.assembly_accession

    print(f"Direct search: {assembly_accession}")

    assembly_fasta_path = locate_assembly_fasta(
        assembly_accession
    )
    assembly_sequences = {
        record.id: str(record.seq).upper()
        for record in SeqIO.parse(
            str(assembly_fasta_path),
            "fasta",
        )
    }
    assembly_sequence_cache[assembly_accession] = (
        assembly_sequences
    )

    assembly_blast_path = (
        pilot_work_directory
        / assembly_accession
        / "coding_reference_blast.tsv"
    )

    blast_command = [
        str(blastn_executable),
        "-query",
        str(query_fasta_path),
        "-subject",
        str(assembly_fasta_path),
        "-task",
        "blastn",
        "-dust",
        "no",
        "-soft_masking",
        "false",
        "-evalue",
        "1e-20",
        "-max_target_seqs",
        "50",
        "-max_hsps",
        "1",
        "-num_threads",
        "2",
        "-outfmt",
        (
            "6 qseqid sseqid pident length qlen "
            "qstart qend sstart send bitscore evalue"
        ),
        "-out",
        str(assembly_blast_path),
    ]

    blast_result = subprocess.run(
        blast_command,
        capture_output=True,
        text=True,
    )
    if blast_result.returncode != 0:
        raise RuntimeError(
            f"BLASTN failed for {assembly_accession}:\n"
            f"{blast_result.stderr}"
        )

    if assembly_blast_path.stat().st_size == 0:
        blast_table = pd.DataFrame(columns=blast_columns)
    else:
        blast_table = pd.read_csv(
            assembly_blast_path,
            sep="\t",
            names=blast_columns,
        )

    for column in blast_columns[2:]:
        blast_table[column] = pd.to_numeric(
            blast_table[column],
            errors="coerce",
        )

    blast_table.insert(0, "assembly_accession", assembly_accession)
    blast_table.insert(0, "biosample", biosample)
    blast_table["aligned_reference_bases"] = (
        blast_table["query_stop"]
        - blast_table["query_start"]
    ).abs() + 1
    blast_table["query_coverage"] = (
        100.0
        * blast_table["aligned_reference_bases"]
        / blast_table["reference_length"]
    ).clip(upper=100.0)
    blast_table["passes_threshold"] = (
        (
            blast_table["query_coverage"]
            >= MINIMUM_QUERY_COVERAGE
        )
        & (
            blast_table["percent_identity"]
            >= MINIMUM_NUCLEOTIDE_IDENTITY
        )
    )

    blast_table["accepted_after_review"] = False
    blast_table["review_reason"] = ""

    for reviewed_key, reviewed_rule in (
        REVIEWED_CODING_HITS.items()
    ):
        reviewed_assembly, reviewed_locus = reviewed_key
        if reviewed_assembly != assembly_accession:
            continue

        reviewed_candidates = blast_table[
            blast_table["locus_id"] == reviewed_locus
        ].sort_values(
            "bit_score",
            ascending=False,
        )
        if reviewed_candidates.empty:
            raise ValueError(
                f"The reviewed hit {reviewed_key} was not found."
            )

        reviewed_index = reviewed_candidates.index[0]
        reviewed_hit = blast_table.loc[reviewed_index]

        if (
            reviewed_hit["contig_id"]
            != reviewed_rule["expected_contig"]
            or reviewed_hit["percent_identity"]
            < reviewed_rule["minimum_identity"]
            or reviewed_hit["query_coverage"]
            < reviewed_rule["minimum_coverage"]
        ):
            raise ValueError(
                f"The BLAST evidence for reviewed hit "
                f"{reviewed_key} no longer matches the reviewed "
                "pilot result."
            )

        blast_table.loc[
            reviewed_index,
            "accepted_after_review",
        ] = True
        blast_table.loc[
            reviewed_index,
            "review_reason",
        ] = reviewed_rule["review_reason"]

    blast_table["accepted_hit"] = (
        blast_table["passes_threshold"]
        | blast_table["accepted_after_review"]
    )
    all_blast_tables.append(blast_table.copy())

    accepted_hits = blast_table[
        blast_table["accepted_hit"]
    ].sort_values(
        ["locus_id", "bit_score"],
        ascending=[True, False],
    )

    # Keep one best hit per reference locus for this pilot.
    best_hits = accepted_hits.drop_duplicates(
        subset=["locus_id"],
        keep="first",
    )

    for hit in best_hits.itertuples(index=False):
        locus_id = hit.locus_id
        reference_info = coding_reference_lookup[locus_id]
        extracted = extract_reference_oriented_sequence(
            hit,
            assembly_sequences,
        )
        integrity = assess_coding_integrity(
            extracted["sequence"],
            reference_info["reference_sequence"],
            extracted["touches_contig_edge"],
        )

        direct_sequence_rows.append(
            {
                "biosample": biosample,
                "assembly_accession": assembly_accession,
                "locus_id": locus_id,
                "locus_name": reference_info["locus_name"],
                "sequence_source": (
                    "direct BLASTN against MG1655 reference"
                ),
                **extracted,
                "detection_method": (
                    f"{hit.percent_identity:.2f}% identity; "
                    f"{hit.query_coverage:.2f}% coverage"
                ),
                "acceptance_rule": (
                    "reviewed pilot decision"
                    if hit.accepted_after_review
                    else "standard identity and coverage thresholds"
                ),
                "review_reason": hit.review_reason,
                "percent_identity": float(hit.percent_identity),
                "query_coverage": float(hit.query_coverage),
                "reference_length": int(hit.reference_length),
                **integrity,
            }
        )


direct_sequence_details = pd.DataFrame(direct_sequence_rows)

reviewed_accepted_details = direct_sequence_details[
    direct_sequence_details["acceptance_rule"]
    == "reviewed pilot decision"
].copy()
reviewed_accepted_details[
    "overlap_with_accepted_ompF_bases"
] = 0

for reviewed_index, reviewed_row in (
    reviewed_accepted_details.iterrows()
):
    accepted_ompf_hits = direct_sequence_details[
        (
            direct_sequence_details["assembly_accession"]
            == reviewed_row["assembly_accession"]
        )
        & (
            direct_sequence_details["locus_name"]
            == "ompF"
        )
        & (
            direct_sequence_details["contig_id"]
            == reviewed_row["contig_id"]
        )
    ]

    overlap_bases = 0
    for omp_f_hit in accepted_ompf_hits.itertuples(
        index=False
    ):
        overlap_start = max(
            int(reviewed_row["start"]),
            int(omp_f_hit.start),
        )
        overlap_stop = min(
            int(reviewed_row["stop"]),
            int(omp_f_hit.stop),
        )
        overlap_bases = max(
            overlap_bases,
            max(0, overlap_stop - overlap_start + 1),
        )

    reviewed_accepted_details.loc[
        reviewed_index,
        "overlap_with_accepted_ompF_bases",
    ] = overlap_bases

if (
    reviewed_accepted_details[
        "overlap_with_accepted_ompF_bases"
    ]
    > 0
).any():
    raise ValueError(
        "A manually reviewed ompC hit overlaps the accepted "
        "ompF sequence."
    )

for column in [
    "percent_identity",
    "query_coverage",
    "reference_length",
]:
    retained_sequence_details[column] = np.nan

for column in [
    "acceptance_rule",
    "review_reason",
    "coding_integrity",
    "integrity_reason",
    "length_difference",
    "ambiguous_nucleotides",
    "start_codon",
    "internal_stop_count",
    "terminal_codon",
    "touches_contig_edge",
    "contig_length",
]:
    if column not in retained_sequence_details.columns:
        retained_sequence_details[column] = pd.NA

sequence_details = pd.concat(
    [retained_sequence_details, direct_sequence_details],
    ignore_index=True,
    sort=False,
)
sequence_details = sequence_details.sort_values(
    ["assembly_accession", "locus_id", "contig_id", "start"]
).reset_index(drop=True)
sequence_details["copy_index"] = (
    sequence_details.groupby(
        ["assembly_accession", "locus_id"]
    ).cumcount()
    + 1
)


# Create one summary row per pilot assembly and Model C locus.
detail_counts = (
    sequence_details.groupby(
        ["assembly_accession", "locus_id"]
    )
    .size()
    .to_dict()
)

previous_promoter_status = {
    row.assembly_accession: row.extraction_status
    for row in previous_sequence_summary[
        previous_sequence_summary["locus_name"] == "blaTEMp"
    ].itertuples(index=False)
}

summary_rows = []
for pilot_row in pilot_manifest.itertuples(index=False):
    for locus in reference_audit.itertuples(index=False):
        key = (pilot_row.assembly_accession, locus.locus_id)
        copy_number = int(detail_counts.get(key, 0))

        if copy_number > 0:
            status = "sequence extracted"
        elif locus.locus_name == "blaTEMp":
            status = previous_promoter_status.get(
                pilot_row.assembly_accession,
                "blaTEM not detected",
            )
        elif locus.reference_rule == full_gene_rule:
            status = "not detected by AMRFinderPlus"
        else:
            status = (
                "no direct alignment meeting thresholds"
            )

        summary_rows.append(
            {
                "biosample": pilot_row.biosample,
                "assembly_accession": pilot_row.assembly_accession,
                "locus_id": locus.locus_id,
                "locus_name": locus.locus_name,
                "reference_rule": locus.reference_rule,
                "extraction_status": status,
                "copy_number": copy_number,
            }
        )

pilot_sequence_summary = pd.DataFrame(summary_rows)
assert len(pilot_sequence_summary) == (
    PILOT_ASSEMBLY_COUNT * MODEL_C_LOCUS_COUNT
)


# Overwrite the existing pilot files and add the BLAST result file.
sequence_details.to_csv(sequence_details_path, index=False)
pilot_sequence_summary.to_csv(sequence_summary_path, index=False)

combined_blast_results = pd.concat(
    all_blast_tables,
    ignore_index=True,
)
combined_blast_results.to_csv(blast_results_path, index=False)

with open(sequence_fasta_path, "w", encoding="utf-8") as fasta_file:
    for row in sequence_details.itertuples(index=False):
        fasta_file.write(
            f">{row.biosample}|{row.assembly_accession}|"
            f"{row.locus_id}|{row.locus_name}|"
            f"copy_{row.copy_index}\n"
        )
        sequence = str(row.sequence)
        for start in range(0, len(sequence), 80):
            fasta_file.write(sequence[start:start + 80] + "\n")


# Validate the direct coding-locus results.
coding_summary = pilot_sequence_summary[
    pilot_sequence_summary["reference_rule"]
    == coding_reference_rule
]
expected_coding_results = (
    PILOT_ASSEMBLY_COUNT * len(coding_reference_panel)
)
coding_results_found = int(
    (coding_summary["copy_number"] > 0).sum()
)
missing_coding_results = coding_summary[
    coding_summary["copy_number"] == 0
]

pilot_full_gene_sequences = int(
    (
        sequence_details["sequence_source"]
        == "AMRFinderPlus nucleotide hit"
    ).sum()
)
pilot_amrfinder_rows = len(
    combined_amrfinder_results
)

# For every missing coding locus, retain its highest-scoring BLAST alignment
# even when that alignment does not pass the fixed thresholds. This table is
# evidence for review; it does not automatically accept the alignment.
missing_coding_keys = missing_coding_results[
    ["biosample", "assembly_accession", "locus_id", "locus_name"]
].copy()

missing_candidate_hits = missing_coding_keys.merge(
    combined_blast_results,
    on=["biosample", "assembly_accession", "locus_id"],
    how="left",
)
missing_candidate_hits = missing_candidate_hits.sort_values(
    ["biosample", "assembly_accession", "locus_id", "bit_score"],
    ascending=[True, True, True, False],
    na_position="last",
)
best_missing_candidate_hits = missing_candidate_hits.drop_duplicates(
    subset=["biosample", "assembly_accession", "locus_id"],
    keep="first",
).copy()

best_missing_candidate_hits["identity_result"] = np.where(
    best_missing_candidate_hits["percent_identity"]
    >= MINIMUM_NUCLEOTIDE_IDENTITY,
    "passes",
    "below threshold",
)
best_missing_candidate_hits["coverage_result"] = np.where(
    best_missing_candidate_hits["query_coverage"]
    >= MINIMUM_QUERY_COVERAGE,
    "passes",
    "below threshold",
)

# Extract each best missing candidate only for integrity review. These
# sequences are not added to the accepted pilot sequence records.
candidate_integrity_rows = []
for candidate in best_missing_candidate_hits.itertuples(
    index=False
):
    if pd.isna(candidate.contig_id):
        continue

    reference_info = coding_reference_lookup[
        candidate.locus_id
    ]
    assembly_sequences = assembly_sequence_cache[
        candidate.assembly_accession
    ]
    extracted = extract_reference_oriented_sequence(
        candidate,
        assembly_sequences,
    )
    integrity = assess_coding_integrity(
        extracted["sequence"],
        reference_info["reference_sequence"],
        extracted["touches_contig_edge"],
    )

    accepted_ompf_hits = direct_sequence_details[
        (
            direct_sequence_details["assembly_accession"]
            == candidate.assembly_accession
        )
        & (
            direct_sequence_details["locus_name"]
            == "ompF"
        )
        & (
            direct_sequence_details["contig_id"]
            == extracted["contig_id"]
        )
    ]

    overlap_bases = 0
    for omp_f_hit in accepted_ompf_hits.itertuples(
        index=False
    ):
        overlap_start = max(
            int(extracted["start"]),
            int(omp_f_hit.start),
        )
        overlap_stop = min(
            int(extracted["stop"]),
            int(omp_f_hit.stop),
        )
        overlap_bases = max(
            overlap_bases,
            max(0, overlap_stop - overlap_start + 1),
        )

    candidate_integrity_rows.append(
        {
            "biosample": candidate.biosample,
            "assembly_accession": (
                candidate.assembly_accession
            ),
            "locus_id": candidate.locus_id,
            "candidate_sequence_length": (
                extracted["sequence_length"]
            ),
            "candidate_start": extracted["start"],
            "candidate_stop": extracted["stop"],
            "candidate_strand": extracted["strand"],
            "candidate_touches_contig_edge": (
                extracted["touches_contig_edge"]
            ),
            "overlap_with_accepted_ompF_bases": (
                overlap_bases
            ),
            **integrity,
        }
    )

candidate_integrity_details = pd.DataFrame(
    candidate_integrity_rows
)
if not candidate_integrity_details.empty:
    best_missing_candidate_hits = (
        best_missing_candidate_hits.merge(
            candidate_integrity_details,
            on=[
                "biosample",
                "assembly_accession",
                "locus_id",
            ],
            how="left",
        )
    )

coding_integrity_summary = (
    direct_sequence_details.groupby(
        "coding_integrity",
        as_index=False,
    )
    .agg(extracted_sequences=("locus_id", "count"))
)
accepted_integrity_review = direct_sequence_details[
    direct_sequence_details["coding_integrity"]
    != "intact ORF"
].copy()

pilot_status = (
    "passed"
    if (
        coding_results_found == expected_coding_results
        and pilot_full_gene_sequences
        == EXPECTED_PILOT_FULL_GENE_SEQUENCES
        and pilot_amrfinder_rows
        == EXPECTED_PILOT_AMRFINDER_ROWS
    )
    else "review required"
)

pilot_validation_summary = pd.DataFrame(
    [
        {
            "metric": "Pilot assemblies",
            "value": PILOT_ASSEMBLY_COUNT,
        },
        {
            "metric": "Expected assembly-locus results",
            "value": PILOT_ASSEMBLY_COUNT * MODEL_C_LOCUS_COUNT,
        },
        {
            "metric": "Observed assembly-locus results",
            "value": len(pilot_sequence_summary),
        },
        {
            "metric": "Extracted sequence records",
            "value": len(sequence_details),
        },
        {
            "metric": "AMRFinderPlus result rows",
            "value": pilot_amrfinder_rows,
        },
        {
            "metric": "Extracted full-gene sequences",
            "value": pilot_full_gene_sequences,
        },
        {
            "metric": "Expected coding-locus results",
            "value": expected_coding_results,
        },
        {
            "metric": "Coding-locus results found",
            "value": coding_results_found,
        },
        {
            "metric": "Coding-locus results missing",
            "value": len(missing_coding_results),
        },
        {
            "metric": "Reviewed coding hits accepted",
            "value": len(reviewed_accepted_details),
        },
        {
            "metric": "Accepted intact coding sequences",
            "value": int(
                (
                    direct_sequence_details[
                        "coding_integrity"
                    ]
                    == "intact ORF"
                ).sum()
            ),
        },
        {
            "metric": (
                "Accepted potentially disrupted "
                "coding sequences"
            ),
            "value": int(
                (
                    direct_sequence_details[
                        "coding_integrity"
                    ]
                    == "potentially disrupted"
                ).sum()
            ),
        },
        {
            "metric": "Accepted uncertain coding sequences",
            "value": int(
                (
                    direct_sequence_details[
                        "coding_integrity"
                    ]
                    == "uncertain"
                ).sum()
            ),
        },
        {
            "metric": "Pilot validation status",
            "value": pilot_status,
        },
    ]
)

status_summary = (
    pilot_sequence_summary.groupby(
        ["reference_rule", "extraction_status"],
        as_index=False,
    )
    .agg(locus_results=("locus_id", "count"))
)

direct_alignment_summary = (
    direct_sequence_details.groupby(
        ["locus_id", "locus_name"],
        as_index=False,
    )
    .agg(
        extracted_sequences=("sequence", "count"),
        minimum_identity=("percent_identity", "min"),
        minimum_coverage=("query_coverage", "min"),
        potentially_disrupted=(
            "coding_integrity",
            lambda values: int(
                (values == "potentially disrupted").sum()
            ),
        ),
        uncertain=(
            "coding_integrity",
            lambda values: int(
                (values == "uncertain").sum()
            ),
        ),
    )
)

saved_files = [
    pilot_manifest_path,
    sequence_details_path,
    sequence_summary_path,
    amrfinder_results_path,
    blast_results_path,
    sequence_fasta_path,
]
file_sizes = pd.DataFrame(
    [
        {
            "file": path.name,
            "size_MB": round(
                path.stat().st_size / (1024 ** 2),
                3,
            ),
        }
        for path in saved_files
    ]
)

display(pilot_validation_summary)
print()
display(status_summary)

if len(reviewed_accepted_details) > 0:
    print("\nReviewed coding hits accepted:")
    display(
        reviewed_accepted_details[
            [
                "biosample",
                "assembly_accession",
                "locus_id",
                "locus_name",
                "contig_id",
                "percent_identity",
                "query_coverage",
                "sequence_length",
                "reference_length",
                "length_difference",
                "coding_integrity",
                "integrity_reason",
                "overlap_with_accepted_ompF_bases",
                "review_reason",
            ]
        ]
    )

if len(missing_coding_results) > 0:
    print("\nCoding loci requiring review:")
    display(
        missing_coding_results[
            [
                "biosample",
                "assembly_accession",
                "locus_id",
                "locus_name",
                "extraction_status",
            ]
        ]
    )
    print(
        "\nBest BLAST alignment for each missing coding locus "
        "(review only; not accepted):"
    )
    display(
        best_missing_candidate_hits[
            [
                "biosample",
                "assembly_accession",
                "locus_id",
                "locus_name",
                "contig_id",
                "percent_identity",
                "identity_result",
                "aligned_reference_bases",
                "reference_length",
                "query_coverage",
                "coverage_result",
                "query_start",
                "query_stop",
                "subject_start",
                "subject_stop",
                "bit_score",
                "e_value",
            ]
        ]
    )
    if not candidate_integrity_details.empty:
        print(
            "\nCoding-integrity and ompF-overlap review "
            "for each missing candidate:"
        )
        display(
            best_missing_candidate_hits[
                [
                    "biosample",
                    "assembly_accession",
                    "locus_id",
                    "locus_name",
                    "candidate_sequence_length",
                    "reference_length",
                    "length_difference",
                    "candidate_start",
                    "candidate_stop",
                    "candidate_strand",
                    "candidate_touches_contig_edge",
                    "start_codon",
                    "internal_stop_count",
                    "terminal_codon",
                    "ambiguous_nucleotides",
                    "coding_integrity",
                    "integrity_reason",
                    "overlap_with_accepted_ompF_bases",
                ]
            ]
        )

print("\nCoding-integrity summary for accepted coding loci:")
display(coding_integrity_summary)

if len(accepted_integrity_review) > 0:
    print(
        "\nAccepted coding sequences requiring "
        "integrity review:"
    )
    display(
        accepted_integrity_review[
            [
                "biosample",
                "assembly_accession",
                "locus_id",
                "locus_name",
                "sequence_length",
                "reference_length",
                "length_difference",
                "start_codon",
                "internal_stop_count",
                "terminal_codon",
                "ambiguous_nucleotides",
                "touches_contig_edge",
                "coding_integrity",
                "integrity_reason",
            ]
        ]
    )

print("\nDirect coding-locus alignment summary:")
display(direct_alignment_summary)
print()
display(file_sizes)

print(
    f"\nTotal saved pilot size: "
    f"{file_sizes['size_MB'].sum():.3f} MB"
)

if pilot_status == "passed":
    print(
        "\nPilot extraction passed. The following "
        "full-extraction cell can now use this method "
        "and save new borderline hits for review."
    )
else:
    print(
        "\nDo not begin the full extraction until the "
        "pilot counts and any displayed review items "
        "have been checked."
    )


Installing AMRFinderPlus 4.2.7...


In [ ]:
#@title Cell 23.11 - Restartable full targeted-sequence extraction
# This cell processes the 9,058 Model C assemblies in deterministic batches.
# Each completed batch is validated, compressed into one ZIP archive, copied
# to Google Drive, and recorded before its downloaded genomes are deleted.
# Rerunning this cell skips every completed batch.

from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import time
import zipfile

import numpy as np
import pandas as pd
from Bio import SeqIO


# ---------------------------------------------------------------------------
# Fixed settings
# ---------------------------------------------------------------------------

EXPECTED_ASSEMBLIES = 9058
EXPECTED_LOCI = 297
EXPECTED_FULL_GENE_LOCI = 266
EXPECTED_CODING_REFERENCE_LOCI = 30

ASSEMBLIES_PER_BATCH = 10
FIRST_RUN_BATCH_LIMIT = 1
LATER_RUN_BATCH_LIMIT = 6
# --- Process without interruption.---#
#PROCESS_ALL_PENDING_BATCHES_AFTER_FIRST = True # Process without interruption.
# ------------------------------------#
COMMAND_RETRIES = 3
THREADS = 2

MINIMUM_NUCLEOTIDE_IDENTITY = 90.0
MINIMUM_QUERY_COVERAGE = 90.0

# ompC is more divergent than the other MG1655 coding loci. A full-length
# ompC hit is accepted at this locus-specific identity threshold only when
# the extracted sequence is not uncertain and does not overlap ompF.
OMPC_MINIMUM_NUCLEOTIDE_IDENTITY = 85.0
OMPC_MINIMUM_QUERY_COVERAGE = 99.0

# A hit below the standard threshold is stored for later review when both
# values meet these lower review limits. It is not used in the kernel until
# reviewed.
BORDERLINE_IDENTITY = 80.0
BORDERLINE_COVERAGE = 80.0

MINIMUM_DRIVE_FREE_GB = 1.0

# These full-length ompC hits were explicitly reviewed. Both have intact
# reading frames, complete reference coverage, no ambiguous nucleotides,
# no contig-edge contact, and no overlap with the accepted ompF sequence.
REVIEWED_CODING_HITS = {
    ("GCA_000522325.1", "LOCUS_0281"): {
        "expected_contig": "KI929782.1",
        "minimum_identity": 89.0,
        "minimum_coverage": 99.0,
        "review_reason": (
            "reviewed full-length ompC hit; distinct from ompF"
        ),
    },
    ("GCA_000692875.1", "LOCUS_0281"): {
        "expected_contig": "KK736562.1",
        "minimum_identity": 89.5,
        "minimum_coverage": 99.0,
        "review_reason": (
            "reviewed full-length ompC hit; intact ORF and "
            "distinct from ompF"
        ),
    }
}

WORK_DIRECTORY = Path("/content/notebook23_work")
NOTEBOOK23_DIRECTORY = Path(
    "/content/drive/MyDrive/Model3_MIC_Project/notebook23"
)
FULL_OUTPUT_DIRECTORY = (
    NOTEBOOK23_DIRECTORY / "full_extraction"
)
BATCH_OUTPUT_DIRECTORY = FULL_OUTPUT_DIRECTORY / "batches"
FULL_WORK_DIRECTORY = WORK_DIRECTORY / "full_extraction"

MANIFEST_CANDIDATES = [
    WORK_DIRECTORY
    / "results"
    / "23_model_c_assembly_manifest.csv",
    NOTEBOOK23_DIRECTORY
    / "23_model_c_assembly_manifest.csv",
]
REFERENCE_AUDIT_PATH = (
    NOTEBOOK23_DIRECTORY
    / "23_target_amr_reference_audit.csv"
)
PILOT_SEQUENCE_PATH = (
    NOTEBOOK23_DIRECTORY
    / "pilot_extraction"
    / "23_pilot_target_sequences.csv"
)
PROGRESS_PATH = (
    FULL_OUTPUT_DIRECTORY
    / "23_full_extraction_progress.csv"
)
OMPC_REBUILD_MARKER_PATH = (
    FULL_OUTPUT_DIRECTORY
    / "23_ompC_general_rule_rebuild_started.txt"
)
OMPC_BACKUP_DIRECTORY = (
    FULL_OUTPUT_DIRECTORY
    / "reviewed_backups"
    / "before_general_ompC_rule"
)

for directory in [
    FULL_OUTPUT_DIRECTORY,
    BATCH_OUTPUT_DIRECTORY,
    FULL_WORK_DIRECTORY,
]:
    directory.mkdir(parents=True, exist_ok=True)

# This one-time migration preserves the earlier archives and then makes
# Batch 0001 the first pending batch. The marker is stored on Google Drive,
# so later reruns continue normally and do not move rebuilt archives.
if not OMPC_REBUILD_MARKER_PATH.exists():
    existing_archives = sorted(
        BATCH_OUTPUT_DIRECTORY.glob("23_batch_*.zip")
    )
    if existing_archives or PROGRESS_PATH.exists():
        OMPC_BACKUP_DIRECTORY.mkdir(
            parents=True,
            exist_ok=True,
        )

        for archive_path in existing_archives:
            backup_path = (
                OMPC_BACKUP_DIRECTORY / archive_path.name
            )
            if backup_path.exists():
                raise FileExistsError(
                    "The ompC-rule backup already contains "
                    f"{backup_path.name}. No files were overwritten."
                )
            shutil.move(
                str(archive_path),
                str(backup_path),
            )

        if PROGRESS_PATH.exists():
            backup_progress_path = (
                OMPC_BACKUP_DIRECTORY
                / PROGRESS_PATH.name
            )
            if backup_progress_path.exists():
                raise FileExistsError(
                    "The ompC-rule backup already contains the "
                    "previous progress file. No file was overwritten."
                )
            shutil.move(
                str(PROGRESS_PATH),
                str(backup_progress_path),
            )

        print(
            "Earlier batch archives were preserved in: "
            f"{OMPC_BACKUP_DIRECTORY}"
        )

    OMPC_REBUILD_MARKER_PATH.write_text(
        "Batch rebuilding started with the general ompC rule.\n",
        encoding="utf-8",
    )
    print(
        "The revised extraction will restart from Batch 0001."
    )


# ---------------------------------------------------------------------------
# Locate and validate inputs and software
# ---------------------------------------------------------------------------

manifest_path = next(
    (
        path
        for path in MANIFEST_CANDIDATES
        if path.exists()
    ),
    None,
)
if manifest_path is None:
    raise FileNotFoundError(
        "The Model C assembly manifest was not found. "
        "Rerun Cell 23.4."
    )

for required_path in [
    REFERENCE_AUDIT_PATH,
    PILOT_SEQUENCE_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required input file was not found: {required_path}"
        )

assembly_manifest = pd.read_csv(manifest_path, dtype=str)
if (
    "assembly_accession"
    not in assembly_manifest.columns
    and "reported_assembly_accession"
    in assembly_manifest.columns
):
    assembly_manifest = assembly_manifest.rename(
        columns={
            "reported_assembly_accession":
                "assembly_accession"
        }
    )

required_manifest_columns = {
    "model_c_row_index",
    "model_3b_row_index",
    "biosample",
    "assembly_accession",
}
missing_manifest_columns = (
    required_manifest_columns
    - set(assembly_manifest.columns)
)
if missing_manifest_columns:
    raise ValueError(
        "Assembly manifest columns are missing: "
        f"{sorted(missing_manifest_columns)}"
    )

assembly_manifest["model_c_row_index"] = pd.to_numeric(
    assembly_manifest["model_c_row_index"],
    errors="raise",
).astype(int)
assembly_manifest["model_3b_row_index"] = pd.to_numeric(
    assembly_manifest["model_3b_row_index"],
    errors="raise",
).astype(int)
assembly_manifest = assembly_manifest.sort_values(
    "model_c_row_index"
).reset_index(drop=True)

assert len(assembly_manifest) == EXPECTED_ASSEMBLIES
assert assembly_manifest["biosample"].is_unique
assert assembly_manifest["assembly_accession"].is_unique
assert assembly_manifest["model_c_row_index"].tolist() == list(
    range(EXPECTED_ASSEMBLIES)
)

reference_audit = pd.read_csv(
    REFERENCE_AUDIT_PATH,
    dtype=str,
)
assert len(reference_audit) == EXPECTED_LOCI

full_gene_rule = (
    "pathogen-specific AMRFinderPlus nucleotide sequence"
)
coding_reference_rule = (
    "E. coli K-12 MG1655 reference gene"
)

full_gene_panel = reference_audit[
    reference_audit["reference_rule"] == full_gene_rule
].copy()
coding_reference_panel = reference_audit[
    reference_audit["reference_rule"]
    == coding_reference_rule
].copy()
bla_temp_panel = reference_audit[
    reference_audit["locus_name"] == "blaTEMp"
].copy()

assert len(full_gene_panel) == EXPECTED_FULL_GENE_LOCI
assert (
    len(coding_reference_panel)
    == EXPECTED_CODING_REFERENCE_LOCI
)
assert len(bla_temp_panel) == 1

if (
    coding_reference_panel["reference_sequence"].isna()
    | coding_reference_panel["reference_sequence"].eq("")
).any():
    raise ValueError(
        "One or more coding-reference sequences are missing."
    )

coding_reference_lookup = (
    coding_reference_panel.set_index("locus_id").to_dict(
        orient="index"
    )
)


def locate_executable(name, candidate_paths):
    candidates = [Path(path) for path in candidate_paths]
    system_path = shutil.which(name)
    if system_path:
        candidates.append(Path(system_path))

    executable = next(
        (path for path in candidates if path.exists()),
        None,
    )
    if executable is None:
        raise FileNotFoundError(
            f"{name} was not found. Rerun the software setup cells."
        )
    return executable


datasets_executable = locate_executable(
    "datasets",
    [WORK_DIRECTORY / "bin" / "datasets"],
)
amrfinder_executable = locate_executable(
    "amrfinder",
    [
        WORK_DIRECTORY
        / "amrfinder_environment"
        / "bin"
        / "amrfinder"
    ],
)
blastn_executable = locate_executable(
    "blastn",
    [
        WORK_DIRECTORY
        / "amrfinder_environment"
        / "bin"
        / "blastn"
    ],
)

command_environment = os.environ.copy()
software_bin_directory = str(amrfinder_executable.parent)
command_environment["PATH"] = (
    software_bin_directory
    + os.pathsep
    + command_environment.get("PATH", "")
)

datasets_version = subprocess.run(
    [str(datasets_executable), "--version"],
    capture_output=True,
    text=True,
    check=True,
    env=command_environment,
).stdout.strip()

amrfinder_version_result = subprocess.run(
    [str(amrfinder_executable), "--version"],
    capture_output=True,
    text=True,
    check=True,
    env=command_environment,
)
amrfinder_version = (
    amrfinder_version_result.stdout.strip()
    or amrfinder_version_result.stderr.strip()
)

blastn_version = subprocess.run(
    [str(blastn_executable), "-version"],
    capture_output=True,
    text=True,
    check=True,
    env=command_environment,
).stdout.splitlines()[0]

database_version_result = subprocess.run(
    [str(amrfinder_executable), "--database_version"],
    capture_output=True,
    text=True,
    check=True,
    env=command_environment,
)
amrfinder_database_version = (
    database_version_result.stdout.strip()
    or database_version_result.stderr.strip()
)

print(datasets_version)
print(f"AMRFinderPlus: {amrfinder_version}")
print(blastn_version)


# ---------------------------------------------------------------------------
# Reference queries and reusable sequence functions
# ---------------------------------------------------------------------------

query_fasta_path = (
    FULL_WORK_DIRECTORY / "23_coding_reference_queries.fasta"
)
with open(query_fasta_path, "w", encoding="utf-8") as fasta_file:
    for reference in coding_reference_panel.itertuples(
        index=False
    ):
        sequence = str(reference.reference_sequence).upper()
        fasta_file.write(f">{reference.locus_id}\n")
        for start in range(0, len(sequence), 80):
            fasta_file.write(
                sequence[start:start + 80] + "\n"
            )


def reverse_complement(sequence):
    table = str.maketrans(
        "ACGTRYKMSWBDHVNacgtrykmswbdhvn",
        "TGCAYRMKSWVHDBNtgcayrmkswvhdbn",
    )
    return sequence.translate(table)[::-1]


STOP_CODONS = {"TAA", "TAG", "TGA"}
BACTERIAL_START_CODONS = {
    "ATG",
    "GTG",
    "TTG",
    "CTG",
    "ATT",
    "ATC",
    "ATA",
}
IUPAC_NUCLEOTIDES = set(
    "ACGTRYSWKMBDHVN"
)


def assess_coding_integrity(
    sequence,
    reference_sequence,
    touches_contig_edge,
):
    sequence = str(sequence).upper()
    reference_sequence = str(reference_sequence).upper()

    ambiguous_nucleotides = sum(
        nucleotide not in {"A", "C", "G", "T"}
        for nucleotide in sequence
    )
    length_difference = (
        len(sequence) - len(reference_sequence)
    )
    length_is_multiple_of_three = len(sequence) % 3 == 0

    complete_codons = [
        sequence[position:position + 3]
        for position in range(0, len(sequence) - 2, 3)
    ]
    start_codon = (
        complete_codons[0] if complete_codons else ""
    )
    terminal_codon = (
        complete_codons[-1] if complete_codons else ""
    )
    internal_stop_count = sum(
        codon in STOP_CODONS
        for codon in complete_codons[:-1]
    )

    reference_has_terminal_stop = (
        len(reference_sequence) >= 3
        and reference_sequence[-3:] in STOP_CODONS
    )
    recognised_start = (
        start_codon in BACTERIAL_START_CODONS
    )
    terminal_stop_present = (
        terminal_codon in STOP_CODONS
    )

    uncertainty_reasons = []
    disruption_reasons = []

    if touches_contig_edge:
        uncertainty_reasons.append(
            "sequence reaches a contig boundary"
        )
    if ambiguous_nucleotides > 0:
        uncertainty_reasons.append(
            f"{ambiguous_nucleotides} ambiguous nucleotide(s)"
        )
    if not length_is_multiple_of_three:
        disruption_reasons.append(
            "length is not a multiple of three"
        )
    if internal_stop_count > 0:
        disruption_reasons.append(
            f"{internal_stop_count} internal stop codon(s)"
        )
    if not recognised_start:
        disruption_reasons.append(
            f"unrecognised start codon {start_codon or 'none'}"
        )
    if (
        reference_has_terminal_stop
        and not terminal_stop_present
    ):
        disruption_reasons.append(
            "terminal stop codon is absent"
        )

    if uncertainty_reasons:
        coding_integrity = "uncertain"
        integrity_reason = "; ".join(
            uncertainty_reasons + disruption_reasons
        )
    elif disruption_reasons:
        coding_integrity = "potentially disrupted"
        integrity_reason = "; ".join(disruption_reasons)
    else:
        coding_integrity = "intact ORF"
        if length_difference == 0:
            integrity_reason = "no coding disruption detected"
        else:
            integrity_reason = (
                "in-frame length difference of "
                f"{length_difference:+d} nucleotide(s)"
            )

    return {
        "coding_integrity": coding_integrity,
        "integrity_reason": integrity_reason,
        "length_difference": length_difference,
        "ambiguous_nucleotides":
            ambiguous_nucleotides,
        "start_codon": start_codon,
        "internal_stop_count": internal_stop_count,
        "terminal_codon": terminal_codon,
    }


def clean_column_name(column_name):
    cleaned = re.sub(
        r"[^a-z0-9]+",
        "_",
        str(column_name).strip().lower(),
    ).strip("_")
    return cleaned


def find_column(table, aliases, required=True):
    normalized_lookup = {
        clean_column_name(column): column
        for column in table.columns
    }
    for alias in aliases:
        normalized_alias = clean_column_name(alias)
        if normalized_alias in normalized_lookup:
            return normalized_lookup[normalized_alias]

    if required:
        raise ValueError(
            "Required AMRFinderPlus column was not found. "
            f"Expected one of {aliases}; observed "
            f"{list(table.columns)}"
        )
    return None


def safe_float(value):
    try:
        if pd.isna(value):
            return np.nan
        return float(value)
    except (TypeError, ValueError):
        return np.nan


def resolve_contig_id(reported_contig_id, assembly_sequences):
    reported = str(reported_contig_id).strip()
    if reported in assembly_sequences:
        return reported

    variants = {
        reported,
        reported.removeprefix("lcl|"),
        reported.split()[0],
    }
    for contig_id in assembly_sequences:
        contig_variants = {
            contig_id,
            contig_id.removeprefix("lcl|"),
            contig_id.split()[0],
        }
        if variants & contig_variants:
            return contig_id

    raise KeyError(
        f"Contig {reported_contig_id} was not found "
        "in the assembly FASTA."
    )


def extract_genomic_interval(
    assembly_sequences,
    contig_id,
    start,
    stop,
    strand,
):
    resolved_contig = resolve_contig_id(
        contig_id,
        assembly_sequences,
    )
    contig_sequence = assembly_sequences[resolved_contig]

    left = min(int(start), int(stop))
    right = max(int(start), int(stop))
    left = max(1, left)
    right = min(len(contig_sequence), right)

    sequence = contig_sequence[left - 1:right]
    normalized_strand = str(strand).strip()
    if normalized_strand == "-" or int(start) > int(stop):
        sequence = reverse_complement(sequence)
        normalized_strand = "-"
    else:
        normalized_strand = "+"

    return {
        "contig_id": resolved_contig,
        "contig_length": len(contig_sequence),
        "start": left,
        "stop": right,
        "strand": normalized_strand,
        "sequence": sequence,
        "sequence_length": len(sequence),
        "touches_contig_edge": (
            left == 1 or right == len(contig_sequence)
        ),
    }


def extract_reference_oriented_sequence(
    hit,
    assembly_sequences,
):
    contig_id = resolve_contig_id(
        hit["contig_id"],
        assembly_sequences,
    )
    contig_sequence = assembly_sequences[contig_id]

    query_start = int(hit["query_start"])
    query_stop = int(hit["query_stop"])
    reference_length = int(hit["reference_length"])
    subject_start = int(hit["subject_start"])
    subject_stop = int(hit["subject_stop"])

    if subject_start <= subject_stop:
        strand = "+"
        extraction_start = subject_start - (query_start - 1)
        extraction_stop = (
            subject_stop + (reference_length - query_stop)
        )
    else:
        strand = "-"
        extraction_start = (
            subject_stop
            - (reference_length - query_stop)
        )
        extraction_stop = (
            subject_start + (query_start - 1)
        )

    unclipped_start = extraction_start
    unclipped_stop = extraction_stop
    extraction_start = max(1, extraction_start)
    extraction_stop = min(
        len(contig_sequence),
        extraction_stop,
    )
    touches_contig_edge = (
        unclipped_start < 1
        or unclipped_stop > len(contig_sequence)
        or extraction_start == 1
        or extraction_stop == len(contig_sequence)
    )

    sequence = contig_sequence[
        extraction_start - 1:extraction_stop
    ]
    if strand == "-":
        sequence = reverse_complement(sequence)

    return {
        "contig_id": contig_id,
        "contig_length": len(contig_sequence),
        "start": extraction_start,
        "stop": extraction_stop,
        "strand": strand,
        "sequence": sequence,
        "sequence_length": len(sequence),
        "touches_contig_edge": touches_contig_edge,
    }


# ---------------------------------------------------------------------------
# AMRFinderPlus-to-panel mapping
# ---------------------------------------------------------------------------

def label_base(label):
    return str(label).split("=", 1)[0].strip().casefold()


full_gene_records = full_gene_panel.to_dict(
    orient="records"
)
full_gene_by_exact = {}
full_gene_by_base = {}
for record in full_gene_records:
    exact_key = str(record["locus_name"]).strip().casefold()
    base_key = label_base(record["locus_name"])
    full_gene_by_exact.setdefault(exact_key, []).append(record)
    full_gene_by_base.setdefault(base_key, []).append(record)


def choose_full_gene_locus(gene_symbol, report_row):
    symbol = str(gene_symbol).strip()
    exact_key = symbol.casefold()
    base_key = label_base(symbol)

    report_text = " ".join(
        str(value)
        for value in report_row.values
        if not pd.isna(value)
    ).upper()

    qualifier = None
    if (
        "MISTRANSLATION" in report_text
        or "FRAME_SHIFT" in report_text
        or "FRAMESHIFT" in report_text
    ):
        qualifier = "MISTRANSLATION"
    elif "PARTIAL" in report_text:
        qualifier = "PARTIAL"

    base_candidates = full_gene_by_base.get(
        base_key,
        [],
    )

    if qualifier is not None:
        qualified = [
            record
            for record in base_candidates
            if str(record["locus_name"]).upper().endswith(
                f"={qualifier}"
            )
        ]
        if len(qualified) == 1:
            return qualified[0], (
                f"matched {qualifier} AMRFinderPlus call"
            )

    exact_candidates = full_gene_by_exact.get(
        exact_key,
        [],
    )
    if len(exact_candidates) == 1:
        return exact_candidates[0], "exact gene-symbol match"

    unsuffixed = [
        record
        for record in base_candidates
        if "=" not in str(record["locus_name"])
    ]
    if len(unsuffixed) == 1:
        return unsuffixed[0], "base gene-symbol match"

    if len(base_candidates) == 1:
        return base_candidates[0], "unique base-label match"

    if len(base_candidates) == 0:
        return None, "AMR gene is not represented in the 266-locus panel"

    return None, (
        "AMR gene maps to multiple Model 3B labels "
        "and requires review"
    )


# ---------------------------------------------------------------------------
# Assembly download and analysis
# ---------------------------------------------------------------------------

def download_assembly(assembly_accession, assembly_directory):
    if assembly_directory.exists():
        shutil.rmtree(assembly_directory)
    assembly_directory.mkdir(parents=True, exist_ok=True)

    archive_path = (
        assembly_directory / f"{assembly_accession}.zip"
    )
    download_result = subprocess.run(
        [
            str(datasets_executable),
            "download",
            "genome",
            "accession",
            assembly_accession,
            "--include",
            "genome",
            "--filename",
            str(archive_path),
        ],
        capture_output=True,
        text=True,
        env=command_environment,
    )
    if download_result.returncode != 0:
        raise RuntimeError(
            f"NCBI download failed for {assembly_accession}: "
            f"{download_result.stderr.strip()}"
        )
    if not zipfile.is_zipfile(archive_path):
        raise ValueError(
            f"Downloaded file is not a valid ZIP: {archive_path}"
        )

    extracted_directory = assembly_directory / "genome"
    extracted_directory.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path, "r") as archive:
        archive.extractall(extracted_directory)

    fasta_candidates = list(
        extracted_directory.rglob("*_genomic.fna")
    )
    if len(fasta_candidates) != 1:
        raise ValueError(
            f"Expected one genomic FASTA for "
            f"{assembly_accession}; found "
            f"{len(fasta_candidates)}."
        )
    return fasta_candidates[0]


def run_amrfinder(
    biosample,
    assembly_accession,
    assembly_fasta_path,
    assembly_directory,
):
    report_path = assembly_directory / "amrfinder.tsv"
    command = [
        str(amrfinder_executable),
        "--nucleotide",
        str(assembly_fasta_path),
        "--organism",
        "Escherichia",
        "--plus",
        "--threads",
        str(THREADS),
        "--output",
        str(report_path),
    ]
    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        env=command_environment,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"AMRFinderPlus failed for {assembly_accession}: "
            f"{result.stderr.strip()}"
        )

    if (
        not report_path.exists()
        or report_path.stat().st_size == 0
    ):
        return pd.DataFrame()

    report = pd.read_csv(
        report_path,
        sep="\t",
        dtype=str,
    )
    report.columns = [
        clean_column_name(column)
        for column in report.columns
    ]
    report.insert(0, "assembly_accession", assembly_accession)
    report.insert(0, "biosample", biosample)
    report.insert(
        2,
        "amrfinder_row_index",
        range(len(report)),
    )
    return report


def extract_amrfinder_sequences(
    biosample,
    assembly_accession,
    report,
    assembly_sequences,
):
    sequence_rows = []
    review_rows = []

    if report.empty:
        return sequence_rows, review_rows

    gene_column = find_column(
        report,
        [
            "gene_symbol",
            "genesymbol",
            "element_symbol",
            "elementsymbol",
        ],
    )
    contig_column = find_column(
        report,
        ["contig_id", "contig"],
    )
    start_column = find_column(report, ["start"])
    stop_column = find_column(report, ["stop"])
    strand_column = find_column(report, ["strand"])
    type_column = find_column(
        report,
        ["element_type", "type"],
        required=False,
    )
    subtype_column = find_column(
        report,
        ["element_subtype", "subtype"],
        required=False,
    )
    method_column = find_column(
        report,
        ["method"],
        required=False,
    )
    identity_column = find_column(
        report,
        [
            "identity_to_reference",
            "percent_identity_to_reference_sequence",
            "identity_to_reference_sequence",
        ],
        required=False,
    )
    coverage_column = find_column(
        report,
        [
            "coverage_of_reference",
            "percent_coverage_of_reference_sequence",
            "coverage_of_reference_sequence",
        ],
        required=False,
    )

    seen_gene_hits = set()
    seen_bla_tem_hits = set()

    for _, report_row in report.iterrows():
        gene_symbol = str(report_row[gene_column]).strip()
        if (
            gene_symbol == ""
            or gene_symbol.upper() == "NA"
            or pd.isna(report_row[gene_column])
        ):
            continue

        element_type = (
            str(report_row[type_column]).upper()
            if type_column is not None
            else ""
        )
        element_subtype = (
            str(report_row[subtype_column]).upper()
            if subtype_column is not None
            else ""
        )

        if type_column is not None and element_type != "AMR":
            continue
        if "POINT" in element_subtype:
            continue

        coordinate_values = [
            report_row[contig_column],
            report_row[start_column],
            report_row[stop_column],
            report_row[strand_column],
        ]
        if any(
            pd.isna(value) or str(value).strip() in {"", "NA"}
            for value in coordinate_values
        ):
            review_rows.append(
                {
                    "biosample": biosample,
                    "assembly_accession": assembly_accession,
                    "review_type":
                        "AMRFinderPlus coordinates missing",
                    "locus_id": "",
                    "locus_name": gene_symbol,
                    "details":
                        "Reported AMR hit has no extractable coordinates.",
                }
            )
            continue

        contig_id = resolve_contig_id(
            report_row[contig_column],
            assembly_sequences,
        )
        start = int(float(report_row[start_column]))
        stop = int(float(report_row[stop_column]))
        strand = str(report_row[strand_column]).strip()

        physical_key = (
            contig_id,
            min(start, stop),
            max(start, stop),
            strand,
            gene_symbol,
        )

        locus_record, mapping_reason = (
            choose_full_gene_locus(
                gene_symbol,
                report_row,
            )
        )

        if locus_record is not None:
            locus_key = (
                locus_record["locus_id"],
                physical_key,
            )
            if locus_key not in seen_gene_hits:
                seen_gene_hits.add(locus_key)
                extracted = extract_genomic_interval(
                    assembly_sequences,
                    contig_id,
                    start,
                    stop,
                    strand,
                )
                ambiguous_count = sum(
                    nucleotide not in {"A", "C", "G", "T"}
                    for nucleotide in extracted["sequence"]
                )
                sequence_status = (
                    "uncertain"
                    if (
                        extracted["touches_contig_edge"]
                        or ambiguous_count > 0
                    )
                    else "AMRFinderPlus sequence extracted"
                )
                sequence_rows.append(
                    {
                        "biosample": biosample,
                        "assembly_accession":
                            assembly_accession,
                        "locus_id":
                            locus_record["locus_id"],
                        "locus_name":
                            locus_record["locus_name"],
                        "locus_group":
                            locus_record["locus_group"],
                        "sequence_source":
                            "AMRFinderPlus nucleotide hit",
                        **extracted,
                        "accepted_for_kernel":
                            sequence_status
                            != "uncertain",
                        "sequence_status": sequence_status,
                        "coding_integrity":
                            "reported by AMRFinderPlus",
                        "integrity_reason": (
                            str(report_row[method_column])
                            if method_column is not None
                            else ""
                        ),
                        "length_difference": np.nan,
                        "ambiguous_nucleotides":
                            ambiguous_count,
                        "start_codon": "",
                        "internal_stop_count": np.nan,
                        "terminal_codon": "",
                        "percent_identity": (
                            safe_float(
                                report_row[identity_column]
                            )
                            if identity_column is not None
                            else np.nan
                        ),
                        "query_coverage": (
                            safe_float(
                                report_row[coverage_column]
                            )
                            if coverage_column is not None
                            else np.nan
                        ),
                        "acceptance_rule":
                            "AMRFinderPlus panel match",
                        "review_reason": mapping_reason,
                    }
                )

                if sequence_status == "uncertain":
                    review_rows.append(
                        {
                            "biosample": biosample,
                            "assembly_accession":
                                assembly_accession,
                            "review_type":
                                "uncertain AMR-gene sequence",
                            "locus_id":
                                locus_record["locus_id"],
                            "locus_name":
                                locus_record["locus_name"],
                            "details": (
                                "Sequence reaches a contig boundary "
                                "or contains ambiguous nucleotides."
                            ),
                        }
                    )
        else:
            review_rows.append(
                {
                    "biosample": biosample,
                    "assembly_accession": assembly_accession,
                    "review_type": "unmapped AMR gene",
                    "locus_id": "",
                    "locus_name": gene_symbol,
                    "details": mapping_reason,
                }
            )

        normalized_symbol = re.sub(
            r"[^a-z0-9]+",
            "",
            gene_symbol.casefold(),
        )
        if normalized_symbol.startswith("blatem"):
            bla_tem_key = (
                contig_id,
                min(start, stop),
                max(start, stop),
                strand,
            )
            if bla_tem_key in seen_bla_tem_hits:
                continue
            seen_bla_tem_hits.add(bla_tem_key)

            contig_sequence = assembly_sequences[contig_id]
            left = min(start, stop)
            right = max(start, stop)

            if strand == "-" or start > stop:
                promoter_start = right + 1
                promoter_stop = right + 100
                promoter_strand = "-"
            else:
                promoter_start = left - 100
                promoter_stop = left - 1
                promoter_strand = "+"

            if (
                promoter_start < 1
                or promoter_stop > len(contig_sequence)
            ):
                review_rows.append(
                    {
                        "biosample": biosample,
                        "assembly_accession":
                            assembly_accession,
                        "review_type":
                            "blaTEMp sequence incomplete",
                        "locus_id":
                            bla_temp_panel.iloc[0]["locus_id"],
                        "locus_name": "blaTEMp",
                        "details": (
                            "The 100-nucleotide upstream region "
                            "crosses a contig boundary."
                        ),
                    }
                )
                continue

            promoter = extract_genomic_interval(
                assembly_sequences,
                contig_id,
                promoter_start,
                promoter_stop,
                promoter_strand,
            )
            ambiguous_count = sum(
                nucleotide not in {"A", "C", "G", "T"}
                for nucleotide in promoter["sequence"]
            )
            sequence_rows.append(
                {
                    "biosample": biosample,
                    "assembly_accession":
                        assembly_accession,
                    "locus_id":
                        bla_temp_panel.iloc[0]["locus_id"],
                    "locus_name": "blaTEMp",
                    "locus_group":
                        bla_temp_panel.iloc[0]["locus_group"],
                    "sequence_source":
                        "100 nucleotides immediately upstream "
                        "of an AMRFinderPlus blaTEM hit",
                    **promoter,
                    "accepted_for_kernel":
                        ambiguous_count == 0,
                    "sequence_status": (
                        "sequence extracted"
                        if ambiguous_count == 0
                        else "uncertain"
                    ),
                    "coding_integrity":
                        "not applicable: promoter",
                    "integrity_reason": "",
                    "length_difference": 0,
                    "ambiguous_nucleotides":
                        ambiguous_count,
                    "start_codon": "",
                    "internal_stop_count": np.nan,
                    "terminal_codon": "",
                    "percent_identity": np.nan,
                    "query_coverage": 100.0,
                    "acceptance_rule":
                        "blaTEM-oriented upstream extraction",
                    "review_reason": "",
                }
            )

    return sequence_rows, review_rows


BLAST_COLUMNS = [
    "locus_id",
    "contig_id",
    "percent_identity",
    "alignment_length",
    "reference_length",
    "query_start",
    "query_stop",
    "subject_start",
    "subject_stop",
    "bit_score",
    "e_value",
]


def run_coding_reference_blast(
    biosample,
    assembly_accession,
    assembly_fasta_path,
    assembly_sequences,
    assembly_directory,
):
    blast_path = assembly_directory / "coding_blast.tsv"
    command = [
        str(blastn_executable),
        "-query",
        str(query_fasta_path),
        "-subject",
        str(assembly_fasta_path),
        "-task",
        "blastn",
        "-dust",
        "no",
        "-soft_masking",
        "false",
        "-evalue",
        "1e-20",
        "-max_target_seqs",
        "50",
        "-max_hsps",
        "1",
        "-num_threads",
        str(THREADS),
        "-outfmt",
        (
            "6 qseqid sseqid pident length qlen "
            "qstart qend sstart send bitscore evalue"
        ),
        "-out",
        str(blast_path),
    ]
    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        env=command_environment,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"BLASTN failed for {assembly_accession}: "
            f"{result.stderr.strip()}"
        )

    if blast_path.stat().st_size == 0:
        blast_table = pd.DataFrame(columns=BLAST_COLUMNS)
    else:
        blast_table = pd.read_csv(
            blast_path,
            sep="\t",
            names=BLAST_COLUMNS,
        )

    for column in BLAST_COLUMNS[2:]:
        blast_table[column] = pd.to_numeric(
            blast_table[column],
            errors="coerce",
        )

    blast_table["aligned_reference_bases"] = (
        blast_table["query_stop"]
        - blast_table["query_start"]
    ).abs() + 1
    blast_table["query_coverage"] = (
        100.0
        * blast_table["aligned_reference_bases"]
        / blast_table["reference_length"]
    ).clip(upper=100.0)
    blast_table["passes_threshold"] = (
        (
            blast_table["percent_identity"]
            >= MINIMUM_NUCLEOTIDE_IDENTITY
        )
        & (
            blast_table["query_coverage"]
            >= MINIMUM_QUERY_COVERAGE
        )
    )

    sequence_rows = []
    best_hit_rows = []
    review_rows = []

    for locus in coding_reference_panel.itertuples(
        index=False
    ):
        locus_hits = blast_table[
            blast_table["locus_id"] == locus.locus_id
        ].sort_values(
            "bit_score",
            ascending=False,
        )

        accepted_hit = None
        acceptance_rule = ""
        review_reason = ""

        standard_hits = locus_hits[
            locus_hits["passes_threshold"]
        ]
        if not standard_hits.empty:
            accepted_hit = standard_hits.iloc[0]
            acceptance_rule = (
                "standard identity and coverage thresholds"
            )
        else:
            reviewed_key = (
                assembly_accession,
                locus.locus_id,
            )
            if reviewed_key in REVIEWED_CODING_HITS:
                if locus_hits.empty:
                    raise ValueError(
                        f"Reviewed hit {reviewed_key} is missing."
                    )
                reviewed_rule = REVIEWED_CODING_HITS[
                    reviewed_key
                ]
                reviewed_hit = locus_hits.iloc[0]
                if (
                    reviewed_hit["contig_id"]
                    != reviewed_rule["expected_contig"]
                    or reviewed_hit["percent_identity"]
                    < reviewed_rule["minimum_identity"]
                    or reviewed_hit["query_coverage"]
                    < reviewed_rule["minimum_coverage"]
                ):
                    raise ValueError(
                        f"Reviewed hit {reviewed_key} no longer "
                        "matches its reviewed evidence."
                    )
                accepted_hit = reviewed_hit
                acceptance_rule = "reviewed pilot decision"
                review_reason = reviewed_rule[
                    "review_reason"
                ]

            if (
                accepted_hit is None
                and locus.locus_name == "ompC"
                and not locus_hits.empty
            ):
                ompC_hit = locus_hits.iloc[0]
                if (
                    ompC_hit["percent_identity"]
                    >= OMPC_MINIMUM_NUCLEOTIDE_IDENTITY
                    and ompC_hit["query_coverage"]
                    >= OMPC_MINIMUM_QUERY_COVERAGE
                ):
                    accepted_hit = ompC_hit
                    acceptance_rule = (
                        "full-length ompC locus-specific thresholds"
                    )
                    review_reason = (
                        "full-length ompC candidate; final acceptance "
                        "requires no sequence uncertainty and no "
                        "overlap with ompF"
                    )

        decision = "missing"
        selected_hit = None

        if accepted_hit is not None:
            decision = "accepted"
            selected_hit = accepted_hit
        elif not locus_hits.empty:
            selected_hit = locus_hits.iloc[0]
            if (
                selected_hit["percent_identity"]
                >= BORDERLINE_IDENTITY
                and selected_hit["query_coverage"]
                >= BORDERLINE_COVERAGE
            ):
                decision = "borderline review"
            else:
                decision = "below review limits"

        if selected_hit is None:
            best_hit_rows.append(
                {
                    "biosample": biosample,
                    "assembly_accession":
                        assembly_accession,
                    "locus_id": locus.locus_id,
                    "locus_name": locus.locus_name,
                    "decision": "no BLAST alignment",
                }
            )
            review_rows.append(
                {
                    "biosample": biosample,
                    "assembly_accession":
                        assembly_accession,
                    "review_type":
                        "coding locus not recovered",
                    "locus_id": locus.locus_id,
                    "locus_name": locus.locus_name,
                    "details": "No BLAST alignment was found.",
                }
            )
            continue

        best_hit_record = {
            "biosample": biosample,
            "assembly_accession": assembly_accession,
            "locus_id": locus.locus_id,
            "locus_name": locus.locus_name,
            **{
                column: selected_hit[column]
                for column in BLAST_COLUMNS[1:]
            },
            "aligned_reference_bases":
                selected_hit["aligned_reference_bases"],
            "query_coverage":
                selected_hit["query_coverage"],
            "passes_threshold":
                selected_hit["passes_threshold"],
            "decision": decision,
            "acceptance_rule": acceptance_rule,
            "review_reason": review_reason,
        }
        best_hit_rows.append(best_hit_record)

        extracted = extract_reference_oriented_sequence(
            selected_hit,
            assembly_sequences,
        )
        integrity = assess_coding_integrity(
            extracted["sequence"],
            locus.reference_sequence,
            extracted["touches_contig_edge"],
        )

        accepted_for_kernel = (
            decision == "accepted"
            and integrity["coding_integrity"] != "uncertain"
        )
        sequence_status = (
            "sequence extracted"
            if accepted_for_kernel
            else (
                "uncertain"
                if (
                    decision == "accepted"
                    and integrity["coding_integrity"]
                    == "uncertain"
                )
                else decision
            )
        )

        sequence_rows.append(
            {
                "biosample": biosample,
                "assembly_accession": assembly_accession,
                "locus_id": locus.locus_id,
                "locus_name": locus.locus_name,
                "locus_group": locus.locus_group,
                "sequence_source":
                    "direct BLASTN against MG1655 reference",
                **extracted,
                "accepted_for_kernel":
                    accepted_for_kernel,
                "sequence_status": sequence_status,
                **integrity,
                "percent_identity":
                    float(
                        selected_hit["percent_identity"]
                    ),
                "query_coverage":
                    float(
                        selected_hit["query_coverage"]
                    ),
                "reference_length":
                    int(selected_hit["reference_length"]),
                "acceptance_rule": acceptance_rule,
                "review_reason": review_reason,
            }
        )

        if decision != "accepted":
            review_rows.append(
                {
                    "biosample": biosample,
                    "assembly_accession":
                        assembly_accession,
                    "review_type":
                        "coding-locus alignment review",
                    "locus_id": locus.locus_id,
                    "locus_name": locus.locus_name,
                    "details": (
                        f"{decision}: "
                        f"{selected_hit['percent_identity']:.3f}% "
                        "identity; "
                        f"{selected_hit['query_coverage']:.3f}% "
                        "reference coverage"
                    ),
                }
            )

        if integrity["coding_integrity"] in {
            "potentially disrupted",
            "uncertain",
        }:
            review_rows.append(
                {
                    "biosample": biosample,
                    "assembly_accession":
                        assembly_accession,
                    "review_type": (
                        "potential coding disruption"
                        if integrity["coding_integrity"]
                        == "potentially disrupted"
                        else "uncertain coding sequence"
                    ),
                    "locus_id": locus.locus_id,
                    "locus_name": locus.locus_name,
                    "details": integrity["integrity_reason"],
                }
            )

    # ompC and ompF must not be assigned to overlapping genomic intervals.
    accepted_porins = [
        row
        for row in sequence_rows
        if (
            row["locus_name"] in {"ompC", "ompF"}
            and row["accepted_for_kernel"]
        )
    ]
    if len(accepted_porins) == 2:
        first, second = accepted_porins
        if first["contig_id"] == second["contig_id"]:
            overlap_start = max(
                int(first["start"]),
                int(second["start"]),
            )
            overlap_stop = min(
                int(first["stop"]),
                int(second["stop"]),
            )
            overlap_bases = max(
                0,
                overlap_stop - overlap_start + 1,
            )
            if overlap_bases > 0:
                for row in accepted_porins:
                    row["accepted_for_kernel"] = False
                    row["sequence_status"] = (
                        "ompC/ompF overlap requires review"
                    )
                    row["review_reason"] = (
                        f"ompC and ompF overlap by "
                        f"{overlap_bases} nucleotide(s)"
                    )
                review_rows.append(
                    {
                        "biosample": biosample,
                        "assembly_accession":
                            assembly_accession,
                        "review_type":
                            "ompC and ompF overlap",
                        "locus_id": "LOCUS_0281;LOCUS_0282",
                        "locus_name": "ompC;ompF",
                        "details": (
                            f"Accepted intervals overlap by "
                            f"{overlap_bases} nucleotide(s)."
                        ),
                    }
                )

    return sequence_rows, best_hit_rows, review_rows


def process_assembly(manifest_row, batch_work_directory):
    biosample = manifest_row.biosample
    assembly_accession = manifest_row.assembly_accession
    assembly_directory = (
        batch_work_directory / assembly_accession
    )
    started = time.perf_counter()

    assembly_fasta_path = download_assembly(
        assembly_accession,
        assembly_directory,
    )
    assembly_sequences = {
        record.id: str(record.seq).upper()
        for record in SeqIO.parse(
            str(assembly_fasta_path),
            "fasta",
        )
    }
    if not assembly_sequences:
        raise ValueError(
            f"No genomic sequences were read for "
            f"{assembly_accession}."
        )

    amrfinder_report = run_amrfinder(
        biosample,
        assembly_accession,
        assembly_fasta_path,
        assembly_directory,
    )
    amr_sequences, amr_reviews = (
        extract_amrfinder_sequences(
            biosample,
            assembly_accession,
            amrfinder_report,
            assembly_sequences,
        )
    )
    coding_sequences, coding_hits, coding_reviews = (
        run_coding_reference_blast(
            biosample,
            assembly_accession,
            assembly_fasta_path,
            assembly_sequences,
            assembly_directory,
        )
    )

    sequence_rows = amr_sequences + coding_sequences
    review_rows = amr_reviews + coding_reviews

    accepted_sequences = sum(
        bool(row["accepted_for_kernel"])
        for row in sequence_rows
    )
    disrupted_sequences = sum(
        row.get("coding_integrity")
        == "potentially disrupted"
        for row in sequence_rows
    )
    uncertain_sequences = sum(
        row.get("coding_integrity") == "uncertain"
        or row.get("sequence_status") == "uncertain"
        for row in sequence_rows
    )

    status_row = {
        "model_c_row_index":
            int(manifest_row.model_c_row_index),
        "model_3b_row_index":
            int(manifest_row.model_3b_row_index),
        "biosample": biosample,
        "assembly_accession": assembly_accession,
        "processing_status": "completed",
        "amrfinder_rows": len(amrfinder_report),
        "sequence_records": len(sequence_rows),
        "accepted_sequence_records": accepted_sequences,
        "potentially_disrupted_sequences":
            disrupted_sequences,
        "uncertain_sequences": uncertain_sequences,
        "review_items": len(review_rows),
        "elapsed_minutes": round(
            (time.perf_counter() - started) / 60.0,
            3,
        ),
        "completed_utc":
            datetime.now(timezone.utc).isoformat(),
    }

    return {
        "status_row": status_row,
        "sequence_rows": sequence_rows,
        "amrfinder_report": amrfinder_report,
        "coding_hit_rows": coding_hits,
        "review_rows": review_rows,
    }


# ---------------------------------------------------------------------------
# Batch archive validation and progress reconstruction
# ---------------------------------------------------------------------------

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as input_file:
        for block in iter(
            lambda: input_file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()


def validate_batch_archive(
    archive_path,
    expected_batch_manifest=None,
):
    if not zipfile.is_zipfile(archive_path):
        raise ValueError(
            f"Invalid batch ZIP archive: {archive_path}"
        )

    with zipfile.ZipFile(archive_path, "r") as archive:
        damaged_member = archive.testzip()
        if damaged_member is not None:
            raise ValueError(
                f"Damaged ZIP member {damaged_member} "
                f"in {archive_path}"
            )

        required_members = {
            "batch_manifest.csv",
            "target_sequences.csv",
            "target_sequences.fasta",
            "amrfinder_results.csv",
            "coding_blast_results.csv",
            "review_items.csv",
            "batch_summary.json",
        }
        missing_members = (
            required_members - set(archive.namelist())
        )
        if missing_members:
            raise ValueError(
                f"Batch archive is missing files: "
                f"{sorted(missing_members)}"
            )

        with archive.open("batch_manifest.csv") as file_handle:
            saved_manifest = pd.read_csv(
                file_handle,
                dtype=str,
            )

        if (
            saved_manifest["processing_status"]
            != "completed"
        ).any():
            raise ValueError(
                f"Batch archive contains incomplete assemblies: "
                f"{archive_path}"
            )

        if expected_batch_manifest is not None:
            expected_accessions = (
                expected_batch_manifest[
                    "assembly_accession"
                ].astype(str).tolist()
            )
            saved_accessions = (
                saved_manifest[
                    "assembly_accession"
                ].astype(str).tolist()
            )
            if saved_accessions != expected_accessions:
                raise ValueError(
                    f"Batch archive accessions do not match "
                    f"the current manifest: {archive_path}"
                )

        with archive.open("batch_summary.json") as file_handle:
            batch_summary = json.load(file_handle)

    return saved_manifest, batch_summary


batch_definitions = []
for start_index in range(
    0,
    EXPECTED_ASSEMBLIES,
    ASSEMBLIES_PER_BATCH,
):
    batch_id = start_index // ASSEMBLIES_PER_BATCH + 1
    batch_manifest = assembly_manifest.iloc[
        start_index:start_index + ASSEMBLIES_PER_BATCH
    ].copy()
    batch_definitions.append(
        (batch_id, batch_manifest)
    )

completed_batch_ids = set()
completed_accessions = set()
archive_by_accession = {}

for batch_id, expected_batch_manifest in batch_definitions:
    archive_path = (
        BATCH_OUTPUT_DIRECTORY
        / f"23_batch_{batch_id:04d}.zip"
    )
    if not archive_path.exists():
        continue

    saved_manifest, _ = validate_batch_archive(
        archive_path,
        expected_batch_manifest,
    )
    completed_batch_ids.add(batch_id)
    for accession in saved_manifest[
        "assembly_accession"
    ].astype(str):
        completed_accessions.add(accession)
        archive_by_accession[accession] = archive_path.name


def write_progress_table():
    progress = assembly_manifest[
        [
            "model_c_row_index",
            "model_3b_row_index",
            "biosample",
            "assembly_accession",
        ]
    ].copy()
    progress["batch_id"] = (
        progress["model_c_row_index"]
        // ASSEMBLIES_PER_BATCH
        + 1
    )
    progress["processing_status"] = np.where(
        progress["assembly_accession"].isin(
            completed_accessions
        ),
        "completed",
        "pending",
    )
    progress["batch_archive"] = (
        progress["assembly_accession"]
        .map(archive_by_accession)
        .fillna("")
    )

    temporary_path = PROGRESS_PATH.with_suffix(
        ".csv.partial"
    )
    progress.to_csv(temporary_path, index=False)
    temporary_path.replace(PROGRESS_PATH)
    return progress


progress_table = write_progress_table()

pending_batches = [
    (batch_id, batch_manifest)
    for batch_id, batch_manifest in batch_definitions
    if batch_id not in completed_batch_ids
]

if not pending_batches:
    print(
        "All 9,058 assemblies have already been processed."
    )
    display(
        pd.DataFrame(
            [
                {
                    "metric": "Completed assemblies",
                    "value": len(completed_accessions),
                },
                {
                    "metric": "Pending assemblies",
                    "value": 0,
                },
                {
                    "metric": "Completed batches",
                    "value": len(completed_batch_ids),
                },
                {
                    "metric": "Extraction status",
                    "value": "complete",
                },
            ]
        )
    )
else:
    batch_limit = (
        FIRST_RUN_BATCH_LIMIT
        if len(completed_batch_ids) == 0
        else LATER_RUN_BATCH_LIMIT
    )

    print(
        f"Completed assemblies: {len(completed_accessions):,}"
    )
    print(
        f"Pending assemblies: "
        f"{EXPECTED_ASSEMBLIES - len(completed_accessions):,}"
    )
    print(
        f"This run will process at most "
        f"{batch_limit} batch(es)."
    )

    run_batch_summaries = []

    for batch_id, batch_manifest in pending_batches[
        :batch_limit
    ]:
        try:
            drive_usage = shutil.disk_usage(
                FULL_OUTPUT_DIRECTORY
            )
            drive_free_gb = (
                drive_usage.free / (1024 ** 3)
            )
        except OSError:
            drive_free_gb = np.nan

        if (
            np.isfinite(drive_free_gb)
            and drive_free_gb > 0
            and drive_free_gb < MINIMUM_DRIVE_FREE_GB
        ):
            raise OSError(
                f"Only {drive_free_gb:.2f} GB is free in "
                "Google Drive. Processing stopped before "
                "starting another batch."
            )

        batch_work_directory = (
            FULL_WORK_DIRECTORY
            / f"batch_{batch_id:04d}"
        )
        if batch_work_directory.exists():
            shutil.rmtree(batch_work_directory)
        batch_work_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        print(
            f"\nStarting batch {batch_id:04d}: "
            f"{len(batch_manifest)} assemblies"
        )

        status_rows = []
        sequence_rows = []
        amrfinder_tables = []
        coding_hit_rows = []
        review_rows = []

        for row_number, manifest_row in enumerate(
            batch_manifest.itertuples(index=False),
            start=1,
        ):
            print(
                f"[{row_number}/{len(batch_manifest)}] "
                f"{manifest_row.assembly_accession}"
            )

            last_error = None
            for attempt in range(1, COMMAND_RETRIES + 1):
                try:
                    result = process_assembly(
                        manifest_row,
                        batch_work_directory,
                    )
                    last_error = None
                    break
                except Exception as error:
                    last_error = error
                    assembly_directory = (
                        batch_work_directory
                        / manifest_row.assembly_accession
                    )
                    if assembly_directory.exists():
                        shutil.rmtree(assembly_directory)

                    if attempt < COMMAND_RETRIES:
                        print(
                            f"  Attempt {attempt} failed: "
                            f"{type(error).__name__}: {error}"
                        )
                        print("  Retrying in 15 seconds.")
                        time.sleep(15)

            if last_error is not None:
                raise RuntimeError(
                    f"Assembly "
                    f"{manifest_row.assembly_accession} failed "
                    f"after {COMMAND_RETRIES} attempts."
                ) from last_error

            status_rows.append(result["status_row"])
            sequence_rows.extend(result["sequence_rows"])
            if not result["amrfinder_report"].empty:
                amrfinder_tables.append(
                    result["amrfinder_report"]
                )
            coding_hit_rows.extend(
                result["coding_hit_rows"]
            )
            review_rows.extend(result["review_rows"])

        batch_status = pd.DataFrame(status_rows)
        batch_sequences = pd.DataFrame(sequence_rows)
        batch_coding_hits = pd.DataFrame(coding_hit_rows)
        batch_reviews = pd.DataFrame(review_rows)
        batch_amrfinder = (
            pd.concat(
                amrfinder_tables,
                ignore_index=True,
                sort=False,
            )
            if amrfinder_tables
            else pd.DataFrame()
        )

        if len(batch_status) != len(batch_manifest):
            raise ValueError(
                "Batch status row count is incomplete."
            )
        if (
            batch_status["processing_status"]
            != "completed"
        ).any():
            raise ValueError(
                "Batch contains an incomplete assembly."
            )

        if not batch_sequences.empty:
            batch_sequences = batch_sequences.sort_values(
                [
                    "assembly_accession",
                    "locus_id",
                    "contig_id",
                    "start",
                ]
            ).reset_index(drop=True)
            batch_sequences["copy_index"] = (
                batch_sequences.groupby(
                    ["assembly_accession", "locus_id"]
                ).cumcount()
                + 1
            )
            batch_sequences["sequence_id"] = (
                batch_sequences["biosample"]
                + "|"
                + batch_sequences["assembly_accession"]
                + "|"
                + batch_sequences["locus_id"]
                + "|copy_"
                + batch_sequences["copy_index"].astype(str)
            )

            invalid_sequences = batch_sequences[
                ~batch_sequences["sequence"]
                .astype(str)
                .str.upper()
                .map(
                    lambda sequence: (
                        set(sequence)
                        <= IUPAC_NUCLEOTIDES
                    )
                )
            ]
            if len(invalid_sequences) > 0:
                raise ValueError(
                    "Extracted sequences contain invalid "
                    "nucleotide symbols."
                )

        # The first batch must reproduce the already validated pilot
        # sequences for its three pilot assemblies.
        pilot_sequences = pd.read_csv(
            PILOT_SEQUENCE_PATH,
            dtype=str,
        )
        pilot_accessions = set(
            pilot_sequences["assembly_accession"]
        )
        batch_pilot_accessions = (
            set(batch_manifest["assembly_accession"])
            & pilot_accessions
        )

        if batch_pilot_accessions:
            expected_counter = Counter(
                (
                    str(row.assembly_accession),
                    str(row.locus_id),
                    str(row.sequence).upper(),
                )
                for row in pilot_sequences[
                    pilot_sequences[
                        "assembly_accession"
                    ].isin(batch_pilot_accessions)
                ].itertuples(index=False)
            )
            observed_counter = Counter(
                (
                    str(row.assembly_accession),
                    str(row.locus_id),
                    str(row.sequence).upper(),
                )
                for row in batch_sequences[
                    batch_sequences[
                        "assembly_accession"
                    ].isin(batch_pilot_accessions)
                ].itertuples(index=False)
            )

            if expected_counter != observed_counter:
                missing_records = (
                    expected_counter - observed_counter
                )
                unexpected_records = (
                    observed_counter - expected_counter
                )
                print(
                    "\nPilot comparison failed."
                )
                print(
                    f"Missing validated sequence records: "
                    f"{sum(missing_records.values())}"
                )
                print(
                    f"Unexpected new sequence records: "
                    f"{sum(unexpected_records.values())}"
                )

                missing_loci = Counter(
                    key[1]
                    for key, count in missing_records.items()
                    for _ in range(count)
                )
                unexpected_loci = Counter(
                    key[1]
                    for key, count
                    in unexpected_records.items()
                    for _ in range(count)
                )
                print(
                    "Missing records by locus:",
                    dict(missing_loci.most_common(20)),
                )
                print(
                    "Unexpected records by locus:",
                    dict(unexpected_loci.most_common(20)),
                )
                raise ValueError(
                    "Full-extraction results do not reproduce "
                    "the validated pilot results."
                )

        batch_files_directory = (
            batch_work_directory / "archive_files"
        )
        batch_files_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        batch_status.to_csv(
            batch_files_directory / "batch_manifest.csv",
            index=False,
        )
        batch_sequences.to_csv(
            batch_files_directory / "target_sequences.csv",
            index=False,
        )
        batch_amrfinder.to_csv(
            batch_files_directory / "amrfinder_results.csv",
            index=False,
        )
        batch_coding_hits.to_csv(
            batch_files_directory
            / "coding_blast_results.csv",
            index=False,
        )
        batch_reviews.to_csv(
            batch_files_directory / "review_items.csv",
            index=False,
        )

        fasta_path = (
            batch_files_directory
            / "target_sequences.fasta"
        )
        with open(
            fasta_path,
            "w",
            encoding="utf-8",
        ) as fasta_file:
            for row in batch_sequences.itertuples(
                index=False
            ):
                fasta_file.write(f">{row.sequence_id}\n")
                sequence = str(row.sequence).upper()
                for start in range(0, len(sequence), 80):
                    fasta_file.write(
                        sequence[start:start + 80] + "\n"
                    )

        batch_summary = {
            "batch_id": batch_id,
            "first_model_c_row_index": int(
                batch_manifest[
                    "model_c_row_index"
                ].min()
            ),
            "last_model_c_row_index": int(
                batch_manifest[
                    "model_c_row_index"
                ].max()
            ),
            "assemblies": len(batch_manifest),
            "sequence_records": len(batch_sequences),
            "accepted_sequence_records": int(
                batch_sequences[
                    "accepted_for_kernel"
                ].astype(bool).sum()
            ),
            "potentially_disrupted_sequences": int(
                (
                    batch_sequences[
                        "coding_integrity"
                    ]
                    == "potentially disrupted"
                ).sum()
            ),
            "uncertain_sequences": int(
                (
                    batch_sequences[
                        "coding_integrity"
                    ]
                    == "uncertain"
                ).sum()
            ),
            "review_items": len(batch_reviews),
            "amrfinder_rows": len(batch_amrfinder),
            "coding_blast_rows": len(
                batch_coding_hits
            ),
            "datasets_version": datasets_version,
            "amrfinder_version": amrfinder_version,
            "amrfinder_database_version":
                amrfinder_database_version,
            "blastn_version": blastn_version,
            "created_utc":
                datetime.now(timezone.utc).isoformat(),
            "validation_status": "passed",
        }
        with open(
            batch_files_directory / "batch_summary.json",
            "w",
            encoding="utf-8",
        ) as summary_file:
            json.dump(
                batch_summary,
                summary_file,
                indent=2,
            )

        local_archive_path = (
            FULL_WORK_DIRECTORY
            / f"23_batch_{batch_id:04d}.zip"
        )
        if local_archive_path.exists():
            local_archive_path.unlink()

        with zipfile.ZipFile(
            local_archive_path,
            "w",
            compression=zipfile.ZIP_DEFLATED,
            compresslevel=6,
        ) as archive:
            for file_path in sorted(
                batch_files_directory.iterdir()
            ):
                archive.write(
                    file_path,
                    arcname=file_path.name,
                )

        validate_batch_archive(
            local_archive_path,
            batch_manifest,
        )
        local_hash = sha256_file(local_archive_path)

        final_archive_path = (
            BATCH_OUTPUT_DIRECTORY
            / local_archive_path.name
        )
        partial_archive_path = final_archive_path.with_suffix(
            ".zip.partial"
        )
        if partial_archive_path.exists():
            partial_archive_path.unlink()
        if final_archive_path.exists():
            raise FileExistsError(
                f"Refusing to overwrite existing batch archive: "
                f"{final_archive_path}"
            )

        shutil.copy2(
            local_archive_path,
            partial_archive_path,
        )
        copied_hash = sha256_file(partial_archive_path)
        if copied_hash != local_hash:
            raise IOError(
                "Google Drive checkpoint hash does not match "
                "the local batch archive."
            )
        partial_archive_path.replace(final_archive_path)
        validate_batch_archive(
            final_archive_path,
            batch_manifest,
        )

        completed_batch_ids.add(batch_id)
        for accession in batch_manifest[
            "assembly_accession"
        ].astype(str):
            completed_accessions.add(accession)
            archive_by_accession[accession] = (
                final_archive_path.name
            )
        progress_table = write_progress_table()

        archive_size_mb = (
            final_archive_path.stat().st_size
            / (1024 ** 2)
        )
        run_batch_summaries.append(
            {
                "batch_id": batch_id,
                "assemblies": len(batch_manifest),
                "sequence_records":
                    len(batch_sequences),
                "accepted_sequences": int(
                    batch_sequences[
                        "accepted_for_kernel"
                    ].astype(bool).sum()
                ),
                "potentially_disrupted": int(
                    (
                        batch_sequences[
                            "coding_integrity"
                        ]
                        == "potentially disrupted"
                    ).sum()
                ),
                "review_items": len(batch_reviews),
                "archive_size_MB": round(
                    archive_size_mb,
                    3,
                ),
                "checkpoint_status": "saved and validated",
            }
        )

        # The archive and progress file are now safe in Google Drive.
        # Delete every downloaded genome and other local batch files.
        shutil.rmtree(batch_work_directory)
        local_archive_path.unlink()

        print(
            f"Batch {batch_id:04d} saved and validated: "
            f"{final_archive_path.name}"
        )
        print(
            "Downloaded genomes for this batch were deleted."
        )

    total_archive_bytes = sum(
        path.stat().st_size
        for path in BATCH_OUTPUT_DIRECTORY.glob(
            "23_batch_*.zip"
        )
    )
    completed_count = len(completed_accessions)
    estimated_total_gb = (
        (
            total_archive_bytes
            / completed_count
            * EXPECTED_ASSEMBLIES
            / (1024 ** 3)
        )
        if completed_count > 0
        else np.nan
    )

    display(pd.DataFrame(run_batch_summaries))
    display(
        pd.DataFrame(
            [
                {
                    "metric": "Completed assemblies",
                    "value": completed_count,
                },
                {
                    "metric": "Pending assemblies",
                    "value": (
                        EXPECTED_ASSEMBLIES
                        - completed_count
                    ),
                },
                {
                    "metric": "Completed batches",
                    "value": len(completed_batch_ids),
                },
                {
                    "metric":
                        "Current saved archive size (GB)",
                    "value": round(
                        total_archive_bytes
                        / (1024 ** 3),
                        3,
                    ),
                },
                {
                    "metric":
                        "Projected final archive size (GB)",
                    "value": round(
                        estimated_total_gb,
                        3,
                    ),
                },
                {
                    "metric": "Progress file",
                    "value": str(PROGRESS_PATH),
                },
            ]
        )
    )

    if completed_count < EXPECTED_ASSEMBLIES:
        print(
            "\nThis run finished safely. Rerun Cell 23.11 "
            "to continue from the next pending batch."
        )
    else:
        print(
            "\nAll 9,058 assemblies have been processed."
        )
